## Publication status — verified completed run

The benchmark run `generation_20261125T164816Z-6faa0a` is complete, and its exported evidence has been checked by recomputing metrics and validating internal consistency. This verification does not independently authenticate GPU execution. See [README.md](README.md) for measured results and limitations.

**Human review remains pending: 0 of 36 responses rated.** Structured scores do not establish written-reply factuality or deployment readiness.

All original notebook cells and outputs follow unchanged. Their statements that the benchmark is pending or still needs execution are historical preparation notes, not the current publication status. This note was added for publication; it is not an execution output.


# Customer-Support Response Fine-Tuning: SFT vs DPO
**Joe Williams · Portfolio project 5 · Revised staged workflow · Full benchmark results pending**

**Question:** Does supervised fine-tuning improve a small chat model's return-policy responses, and does preference optimization add enough value to justify another training stage?

Compare **rules + templates**, **TinyLlama chat before adaptation**, **SFT**, and **SFT → DPO** on the same held-out cases. The business baseline matters: these supplied facts already permit a deterministic decision. A model must justify its additional cost and failure modes.

This notebook contains the code and synthetic data generator; no companion files, private data, API keys, or paid API are required. The model and dependencies require internet access. GPU access and downloads may have costs under your chosen service.

**Run in a fresh Colab session:** Runtime → Change runtime type → T4 GPU; then run the cells in order. Section 6 mounts your Google Drive and saves the run there before training. SFT, DPO, and evaluation are separate cells. No automatic browser download runs during training or evaluation.

Local checks validate mechanics; the revised GPU workflow and a complete benchmark still need execution. Training screenshots from an earlier attempt are not held-out results. If you already have completed adapter downloads from that attempt, preserve them before starting another run.

## 1. Install the fixed dependency versions
Run this before importing the training libraries. Keep Colab's CUDA-enabled PyTorch. If packages were already imported in this session, restart the session after installation and begin again. The notebook records actual package versions and GPU details in the evidence.

In [3]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.6', 'peft==0.18.1', 'accelerate==1.12.0',
    'bitsandbytes==0.48.2', 'numpy==2.3.5'])
import torch
from packaging.version import Version
assert Version(torch.__version__.split('+')[0]) >= Version('2.6'), 'Use PyTorch 2.6 or newer.'
assert torch.cuda.is_available(), 'Select a GPU runtime before training.'
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cu130 | GPU: Tesla T4


## 2. Materialize the project code
This cell writes the embedded source into `support_response_project`. The same files are included in the GitHub starter ZIP. Rerunning this cell restores the original code and configuration; it does not delete prior run folders.

The full training implementation is in `train_evaluate.py`, with the policy, generator, and scorer in `support_data.py`. Later cells display the central functions for discussion.

In [4]:
from pathlib import Path
import os, json, hashlib, sys, importlib
PROJECT = (Path('/content') if Path('/content').is_dir() else Path.cwd()) / 'support_response_project'
PROJECT.mkdir(parents=True, exist_ok=True)
PROJECT_FILES = {'support_data.py': '"""Original synthetic fictional-store cases and strict, limited output metrics."""\nimport collections\nimport hashlib\nimport json\nimport random\n\nCATEGORIES = (\'damaged_item\', \'missing_receipt\', \'outside_window\', \'used_item\',\n              \'eligible\', \'eligible_boundary\')\nKEYS = {\'ticket_id\', \'decision\', \'reason\', \'reply\'}\nREPLIES = {\n    \'damaged_item\': \'Your damaged item needs a support review before a return decision can be made.\',\n    \'missing_receipt\': \'Please provide your receipt so support can review your return request.\',\n    \'outside_window\': \'Your request is outside the stated return window, so this return cannot be approved.\',\n    \'used_item\': \'This return cannot be approved because the policy requires an unused item.\',\n    \'eligible\': \'Your unused item meets the receipt and return-window requirements, so the return is approved.\',\n}\nPOLICY = \'\'\'You draft responses for a fictional store. Use only the supplied facts.\nApply these rules in this exact order:\n1. If damaged is true, decision=review and reason=damaged_item, regardless of other facts.\n2. Otherwise, if receipt is false, decision=review and reason=missing_receipt.\n3. Otherwise, if elapsed_days > return_window_days, decision=deny and reason=outside_window.\n4. Otherwise, if unused is false, decision=deny and reason=used_item.\n5. Otherwise decision=approve and reason=eligible. The deadline day is included.\nReturn one JSON object only with exactly four string fields: ticket_id, decision, reason, reply.\nCopy ticket_id exactly. Write a helpful reply of 1 to 60 words consistent with the decision.\nDo not add markdown, extra keys, promised payments, or unsupported facts.\'\'\'\n\ndef rule_decision(facts):\n    """Explicit business-rule baseline; no learned model is needed for these facts."""\n    if facts[\'damaged\']: return \'review\', \'damaged_item\'\n    if not facts[\'receipt\']: return \'review\', \'missing_receipt\'\n    if facts[\'elapsed_days\'] > facts[\'return_window_days\']: return \'deny\', \'outside_window\'\n    if not facts[\'unused\']: return \'deny\', \'used_item\'\n    return \'approve\', \'eligible\'\n\ndef category(facts):\n    _, reason = rule_decision(facts)\n    if reason == \'eligible\' and facts[\'elapsed_days\'] == facts[\'return_window_days\']:\n        return \'eligible_boundary\'\n    return reason\n\ndef fact_key(facts):\n    # IDs and prose cannot turn identical policy inputs into distinct split groups.\n    return tuple(facts[k] for k in (\'return_window_days\',\'elapsed_days\',\'receipt\',\'unused\',\'damaged\'))\n\ndef rule_response(row):\n    decision, reason = rule_decision(row[\'facts\'])\n    return json.dumps({\'ticket_id\':row[\'id\'],\'decision\':decision,\'reason\':reason,\n                       \'reply\':REPLIES[reason]}, ensure_ascii=False)\n\ndef build_dataset(seed=2026):\n    rng=random.Random(seed)\n    pools={c:[] for c in CATEGORIES}\n    # Complete finite candidate pool; every fact tuple appears once.\n    for window in range(7,181):\n        for elapsed in range(0,201):\n            for receipt in (False,True):\n                for unused in (False,True):\n                    for damaged in (False,True):\n                        f={\'return_window_days\':window,\'elapsed_days\':elapsed,\n                           \'receipt\':receipt,\'unused\':unused,\'damaged\':damaged}\n                        pools[category(f)].append(f)\n    out={s:[] for s in (\'train\',\'development\',\'test\')}\n    leadins=(\'Please handle this return request.\', \'Draft a response for this support ticket.\',\n             \'Apply the store policy to this request.\', \'Respond to the customer using these facts.\')\n    for cat in CATEGORIES:\n        pool=pools[cat];rng.shuffle(pool)\n        for split,start,count in [(\'train\',0,80),(\'development\',80,10),(\'test\',90,20)]:\n            for i,facts in enumerate(pool[start:start+count]):\n                key=json.dumps(fact_key(facts),separators=(\',\',\':\'))\n                ticket=\'T-\'+hashlib.sha256(key.encode()).hexdigest()[:10].upper()\n                row={\'id\':ticket,\'facts\':facts,\'category\':cat,\'split\':split}\n                prompt_facts={\'ticket_id\':ticket,**facts}\n                row[\'prompt\']=POLICY+\'\\n\\n\'+leadins[rng.randrange(len(leadins))]+\'\\n\'+json.dumps(prompt_facts)\n                row[\'completion\']=rule_response(row)\n                bad=json.loads(row[\'completion\']); kind=i%6\n                if kind==0:\n                    bad[\'decision\']={\'approve\':\'deny\',\'deny\':\'review\',\'review\':\'approve\'}[bad[\'decision\']]\n                    error=\'wrong_decision\'\n                elif kind==1:\n                    bad[\'reason\']=\'eligible\' if bad[\'reason\']!=\'eligible\' else \'used_item\';error=\'wrong_reason\'\n                elif kind==2:\n                    bad[\'ticket_id\']=\'T-0000000000\';error=\'wrong_ticket\'\n                elif kind==3:\n                    bad[\'extra\']=\'unrequested\';error=\'extra_key\'\n                elif kind==4:\n                    error=\'markdown_wrapper\'\n                else:\n                    bad[\'reply\']=\'A payment of one hundred dollars is guaranteed to arrive tomorrow.\';error=\'unsupported_reply\'\n                rejected=json.dumps(bad)\n                row[\'rejected\']=\'```json\\n\'+rejected+\'\\n```\' if kind==4 else rejected\n                row[\'rejection_type\']=error\n                out[split].append(row)\n    for rows in out.values(): rng.shuffle(rows)\n    validate_dataset(out)\n    return out\n\ndef validate_dataset(data):\n    all_ids=set(); all_facts=set();all_prompts=set()\n    for split,expected in [(\'train\',80),(\'development\',10),(\'test\',20)]:\n        rows=data[split]\n        if collections.Counter(r[\'category\'] for r in rows)!=collections.Counter({c:expected for c in CATEGORIES}):\n            raise ValueError(\'Unexpected split balance\')\n        for r in rows:\n            if r[\'split\']!=split or r[\'category\']!=category(r[\'facts\']): raise ValueError(\'Split/category mismatch\')\n            key=fact_key(r[\'facts\'])\n            if r[\'id\'] in all_ids or key in all_facts or r[\'prompt\'] in all_prompts: raise ValueError(\'Repeated ID, facts, or prompt\')\n            all_ids.add(r[\'id\']);all_facts.add(key);all_prompts.add(r[\'prompt\'])\n            if r[\'completion\']!=rule_response(r): raise ValueError(\'Reference violates policy\')\n            if r[\'rejected\']==r[\'completion\']: raise ValueError(\'Preference pair is identical\')\n    return {\'rows\':{s:len(r) for s,r in data.items()},\'unique_fact_groups\':len(all_facts),\n            \'shared_fact_groups\':0,\'synthetic\':True,\'wording_policy\':\'Shared fixed templates; new fact combinations, not unseen language.\'}\n\ndef _strict_object(pairs):\n    obj={}\n    for key,value in pairs:\n        if key in obj: raise ValueError(\'Duplicate JSON key\')\n        obj[key]=value\n    return obj\n\ndef score_response(row,text):\n    score={k:False for k in (\'json_valid\',\'schema_valid\',\'decision_correct\',\'reason_correct\',\n                             \'ticket_correct\',\'structured_pass\',\'reference_reply_exact\')}\n    try:\n        value=json.loads(text,object_pairs_hook=_strict_object,\n            parse_constant=lambda x: (_ for _ in ()).throw(ValueError(\'Nonstandard JSON constant\')))\n        score[\'json_valid\']=True\n    except (ValueError,TypeError): return score\n    if not isinstance(value,dict): return score\n    score[\'schema_valid\']=(set(value)==KEYS and all(isinstance(v,str) for v in value.values())\n        and 1<=len(value.get(\'reply\',\'\').split())<=60 and value.get(\'decision\') in (\'approve\',\'deny\',\'review\')\n        and value.get(\'reason\') in REPLIES)\n    truth=json.loads(row[\'completion\'])\n    for field,metric in [(\'decision\',\'decision_correct\'),(\'reason\',\'reason_correct\'),(\'ticket_id\',\'ticket_correct\')]:\n        score[metric]=value.get(field)==truth[field]\n    score[\'structured_pass\']=all(score[k] for k in (\'schema_valid\',\'decision_correct\',\'reason_correct\',\'ticket_correct\'))\n    score[\'reference_reply_exact\']=value.get(\'reply\')==truth[\'reply\']\n    return score\n\ndef summarize(rows,outputs):\n    if len(rows)!=len(outputs) or not rows: raise ValueError(\'Expected aligned nonempty outputs\')\n    scores=[score_response(r,t) for r,t in zip(rows,outputs)]\n    result={\'n_examples\':len(rows)}\n    for name in scores[0]: result[name]=sum(s[name] for s in scores)/len(scores)\n    result[\'per_category\']={c:{\'n_examples\':sum(r[\'category\']==c for r in rows),\n        \'structured_pass\':sum(s[\'structured_pass\'] for r,s in zip(rows,scores) if r[\'category\']==c)/sum(r[\'category\']==c for r in rows)}\n        for c in CATEGORIES if any(r[\'category\']==c for r in rows)}\n    return result\n', 'train_evaluate.py': '"""Fixed SFT/DPO experiment; evidence exports never include model weights.\n\nRun on a fresh CUDA runtime. --smoke uses a tiny RANDOM CPU model and is not a\nbenchmark. Adapter snapshots are backups, not an automatic resume mechanism.\n"""\nimport argparse\nimport csv\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport os\nimport platform\nimport random\nimport re\nimport shutil\nimport time\nimport uuid\nimport zipfile\nfrom collections import Counter\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\n\n\ndef write_json(path, value):\n    atomic_text(path, json.dumps(value, indent=2, allow_nan=False) + "\\n")\n\n\ndef write_jsonl(path, rows):\n    atomic_text(path, "".join(json.dumps(r, ensure_ascii=False, allow_nan=False) + "\\n" for r in rows))\n\n\ndef atomic_text(path, text):\n    """Replace only after the complete new file has been flushed on the same mount."""\n    path = Path(path)\n    temporary = path.with_name(path.name + ".tmp-" + uuid.uuid4().hex)\n    try:\n        with temporary.open("w", encoding="utf-8", newline="") as stream:\n            stream.write(text); stream.flush(); os.fsync(stream.fileno())\n        temporary.replace(path)\n    finally:\n        temporary.unlink(missing_ok=True)\n\n\ndef digest(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef completion_logps(logits, labels):\n    """Sum causal next-token log probabilities; prompt/pad labels are -100."""\n    if logits.ndim != 3 or labels.shape != logits.shape[:2]:\n        raise ValueError("Expected batch x sequence x vocabulary logits and aligned labels")\n    targets = labels[:, 1:]\n    mask = targets != -100\n    logp = F.log_softmax(logits[:, :-1].float(), dim=-1)\n    chosen = logp.gather(-1, targets.masked_fill(~mask, 0).unsqueeze(-1)).squeeze(-1)\n    return (chosen * mask).sum(-1)\n\n\ndef dpo_loss(pi_chosen, pi_rejected, reference_chosen, reference_rejected, beta):\n    """Original sigmoid DPO objective using summed completion log probabilities."""\n    if not math.isfinite(float(beta)) or beta <= 0:\n        raise ValueError("beta must be positive and finite")\n    margin = (pi_chosen - pi_rejected) - (reference_chosen - reference_rejected)\n    return -F.logsigmoid(beta * margin).mean()\n\n\ndef encode_completion(tokenizer, prompt, completion, max_length):\n    prompt_ids = tokenizer.apply_chat_template(\n        [{"role": "user", "content": prompt}], tokenize=True, add_generation_prompt=True)\n    response_ids = tokenizer(completion, add_special_tokens=False)["input_ids"]\n    if tokenizer.eos_token_id is None:\n        raise ValueError("Tokenizer needs an EOS token")\n    if not response_ids or response_ids[-1] != tokenizer.eos_token_id:\n        response_ids.append(tokenizer.eos_token_id)\n    ids = list(prompt_ids) + response_ids\n    if not prompt_ids or len(ids) > max_length:\n        raise ValueError(f"Sequence length {len(ids)} exceeds limit {max_length}; no truncation performed")\n    return {"input_ids": ids, "attention_mask": [1] * len(ids),\n            "labels": [-100] * len(prompt_ids) + response_ids}\n\n\ndef tensor_batch(example, device):\n    # Batch size is one: there is no artificial EOS padding to score.\n    return {k: torch.tensor([v], dtype=torch.long, device=device) for k, v in example.items()}\n\n\ndef forward_logps(model, example, device):\n    batch = tensor_batch(example, device)\n    labels = batch.pop("labels")\n    outputs = model(**batch, use_cache=False)\n    return completion_logps(outputs.logits, labels), int((labels[:, 1:] != -100).sum())\n\n\ndef load_base(config, training=False):\n    from transformers import AutoModelForCausalLM, BitsAndBytesConfig\n    quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",\n        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)\n    model = AutoModelForCausalLM.from_pretrained(config["model_id"], revision=config["model_revision"],\n        quantization_config=quantization, device_map={"": 0}, torch_dtype=torch.float16,\n        attn_implementation="eager", trust_remote_code=False)\n    model.config.use_cache = not training\n    if training:\n        from peft import prepare_model_for_kbit_training\n        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,\n            gradient_checkpointing_kwargs={"use_reentrant": False})\n    return model\n\n\ndef save_adapter(model, tokenizer, directory):\n    directory.mkdir(parents=True, exist_ok=True)\n    model.save_pretrained(directory, safe_serialization=True)\n    tokenizer.save_pretrained(directory)\n    if not (directory / "adapter_model.safetensors").is_file():\n        raise RuntimeError("Adapter weights were not saved")\n    files = {p.name: digest(p) for p in directory.iterdir() if p.is_file() and p.name != "adapter_manifest.json"}\n    write_json(directory / "adapter_manifest.json", files)\n\n\ndef stage_backup(model, tokenizer, run_dir, stage, callback):\n    path = run_dir / "adapters" / stage\n    save_adapter(model, tokenizer, path)\n    archive = run_dir.parent / (run_dir.name + "_" + stage + "_adapter.zip")\n    temporary = Path(shutil.make_archive(str(archive.with_suffix(".tmp-" + uuid.uuid4().hex)),\n        "zip", root_dir=path.parent, base_dir=path.name))\n    temporary.replace(archive)\n    print(f"{stage.upper()} adapter backup: {archive}", flush=True)\n    if callback is not None:\n        callback(stage, run_dir, archive)\n    return {"directory": str(path.relative_to(run_dir)), "zip_name": archive.name,\n            "zip_sha256": digest(archive), "adapter_weights_sha256": digest(path / "adapter_model.safetensors"),\n            "adapter_manifest_sha256": digest(path / "adapter_manifest.json")}\n\n\ndef training_loop(model, tokenizer, encoded, rows, config, run_dir, stage, reference=None):\n    """Fixed optimizer-step budget; deterministic shuffled cycling over training rows."""\n    model.train()\n    for module in model.modules():\n        if isinstance(module, torch.nn.Dropout):\n            module.p = 0.0  # Fixed choice: eliminate policy/reference dropout mismatch.\n    params = [p for p in model.parameters() if p.requires_grad]\n    if not params or any(p.requires_grad and "lora_" not in n for n, p in model.named_parameters()):\n        raise RuntimeError("Only LoRA parameters may be trainable")\n    optimizer = torch.optim.AdamW(params, lr=config["learning_rate_" + stage], weight_decay=0.0)\n    rng = np.random.default_rng(config["seed"] + (1 if stage == "dpo" else 0))\n    order = []; position = 0; logs = []; used_ids = []\n    steps = config[stage + "_steps"]; accumulation = config["gradient_accumulation"]\n    torch.cuda.reset_peak_memory_stats(); started = time.perf_counter()\n    for step in range(1, steps + 1):\n        optimizer.zero_grad(set_to_none=True); losses = []\n        for _ in range(accumulation):\n            if position == len(order):\n                order = rng.permutation(len(rows)).tolist(); position = 0\n            index = order[position]; position += 1; used_ids.append(rows[index]["id"])\n            chosen, n_tokens = forward_logps(model, encoded[index][0], "cuda:0")\n            if stage == "sft":\n                loss = -chosen.mean() / n_tokens\n            else:\n                rejected, _ = forward_logps(model, encoded[index][1], "cuda:0")\n                ref = reference[index]\n                loss = dpo_loss(chosen, rejected,\n                    chosen.new_tensor([ref["chosen_logp"]]), chosen.new_tensor([ref["rejected_logp"]]), config["beta"])\n            if not torch.isfinite(loss):\n                raise FloatingPointError(f"Nonfinite {stage} loss at step {step}")\n            (loss / accumulation).backward(); losses.append(float(loss.detach()))\n        grad_norm = torch.nn.utils.clip_grad_norm_(params, 1.0)\n        if not torch.isfinite(grad_norm):\n            raise FloatingPointError("Nonfinite adapter gradient")\n        optimizer.step()\n        logs.append({"step": step, "loss": float(np.mean(losses)), "gradient_norm_before_clip": float(grad_norm)})\n        if step == 1 or step % 10 == 0 or step == steps:\n            print(f"{stage.upper()} {step}/{steps} loss={logs[-1][\'loss\']:.6f}", flush=True)\n            write_jsonl(run_dir / (stage + "_training_log.jsonl"), logs)\n        if step % config["save_every"] == 0:\n            checkpoint = run_dir / "checkpoints" / f"{stage}_step_{step}"\n            save_adapter(model, tokenizer, checkpoint)\n            write_json(checkpoint / "snapshot.json", {"stage": stage, "optimizer_step": step,\n                "automatic_resume_supported": False})\n    torch.cuda.synchronize()\n    timing = {"seconds": time.perf_counter() - started, "optimizer_steps": steps,\n              "training_example_presentations": len(used_ids),\n              "peak_allocated_gpu_bytes": torch.cuda.max_memory_allocated(),\n              "peak_reserved_gpu_bytes": torch.cuda.max_memory_reserved()}\n    write_jsonl(run_dir / (stage + "_training_log.jsonl"), logs)\n    write_json(run_dir / (stage + "_training_order.json"), used_ids)\n    del optimizer\n    return timing\n\n\ndef reference_scores(model, encoded, rows, run_dir):\n    """Cache the frozen SFT policy BEFORE DPO changes its single adapter."""\n    model.eval(); records = []; started = time.perf_counter()\n    with torch.no_grad():\n        for row, pair in zip(rows, encoded):\n            chosen, _ = forward_logps(model, pair[0], "cuda:0")\n            rejected, _ = forward_logps(model, pair[1], "cuda:0")\n            records.append({"id": row["id"], "chosen_logp": float(chosen), "rejected_logp": float(rejected)})\n    path = run_dir / "dpo_reference_logps.jsonl"; write_jsonl(path, records)\n    return records, {"sha256": digest(path), "seconds": time.perf_counter() - started,\n                     "reference": "Frozen SFT adapter on the same NF4 base", "reduction": "completion token sum including EOS"}\n\n\ndef generate(model, tokenizer, row, config):\n    from transformers import GenerationConfig\n    prompt_ids = tokenizer.apply_chat_template([{"role": "user", "content": row["prompt"]}],\n        tokenize=True, add_generation_prompt=True)\n    if len(prompt_ids) + config["max_new_tokens"] > config["max_length"]:\n        raise ValueError("Generation prompt plus output budget exceeds max_length")\n    ids = torch.tensor([prompt_ids], device="cuda:0"); attention = torch.ones_like(ids)\n    decoding = GenerationConfig(do_sample=False, max_new_tokens=config["max_new_tokens"],\n        use_cache=True, bos_token_id=tokenizer.bos_token_id,\n        pad_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id)\n    torch.cuda.synchronize(); started = time.perf_counter()\n    with torch.inference_mode():\n        output = model.generate(input_ids=ids, attention_mask=attention, generation_config=decoding)\n    torch.cuda.synchronize()\n    elapsed = time.perf_counter() - started; new_ids = output[0, ids.shape[1]:].tolist()\n    return tokenizer.decode(new_ids, skip_special_tokens=True), {\n        "generation_seconds": elapsed, "generated_tokens_including_eos": len(new_ids),\n        "hit_token_limit": len(new_ids) == config["max_new_tokens"] and new_ids[-1] != tokenizer.eos_token_id}\n\n\ndef paired_comparisons(predictions, seed, resamples=2000):\n    by_method = {}\n    for p in predictions:\n        if p["split"] == "test" and p["method"] != "rule_engine":\n            by_method.setdefault(p["method"], {})[p["id"]] = int(p["scores"]["structured_pass"])\n    output = {}\n    for a, b in [("base", "sft"), ("sft", "dpo"), ("base", "dpo")]:\n        if by_method[a].keys() != by_method[b].keys():\n            raise ValueError("Unpaired test rows")\n        differences = np.array([by_method[b][i] - by_method[a][i] for i in sorted(by_method[a])])\n        counts = np.array([(differences == k).sum() for k in [-1, 0, 1]])\n        draws = np.random.default_rng(seed).multinomial(len(differences), counts / len(differences), size=resamples)\n        deltas = (draws[:, 2] - draws[:, 0]) / len(differences)\n        output[b + "_minus_" + a] = {"mean_delta": float(differences.mean()),\n            "ci95": np.quantile(deltas, [.025, .975]).tolist(), "b_better": int(counts[2]),\n            "a_better": int(counts[0]), "ties": int(counts[1]), "resamples": resamples, "seed": seed,\n            "interpretation": "Descriptive paired case bootstrap; no multiplicity or domain-shift adjustment"}\n    return output\n\n\ndef finalize_evidence(run_dir, suffix="results"):\n    # A hard runtime loss can strand a same-directory atomic-write temporary.\n    for candidate in run_dir.rglob("*"):\n        if candidate.is_file() and re.fullmatch(r".+\\.tmp-[0-9a-f]{32}", candidate.name):\n            candidate.unlink()\n    files = sorted(p for p in run_dir.rglob("*") if p.is_file()\n        and p.relative_to(run_dir).parts[0] not in ("adapters", "checkpoints")\n        and p.name != "manifest_sha256.json")\n    write_json(run_dir / "manifest_sha256.json", {str(p.relative_to(run_dir)): digest(p) for p in files})\n    archive = run_dir.parent / (run_dir.name + "_" + suffix + ".zip")\n    temporary = archive.with_name(archive.name + ".tmp-" + uuid.uuid4().hex)\n    try:\n        with zipfile.ZipFile(temporary, "w", zipfile.ZIP_DEFLATED) as z:\n            for p in files + [run_dir / "manifest_sha256.json"]:\n                z.write(p, run_dir.name + "/" + str(p.relative_to(run_dir)))\n        temporary.replace(archive)\n    finally:\n        temporary.unlink(missing_ok=True)\n    print(f"{suffix} evidence ZIP: {archive}", flush=True)\n    return archive\n\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef read_jsonl(path):\n    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]\n\n\ndef require_cuda():\n    if not torch.cuda.is_available():\n        raise RuntimeError("This stage requires CUDA. --smoke validates mechanics on CPU only.")\n\n\ndef environment_info():\n    return {"python": platform.python_version(), "packages": {\n        p: importlib.metadata.version(p) for p in ["torch", "transformers", "peft", "bitsandbytes", "numpy"]},\n        "gpu": torch.cuda.get_device_name(0), "cuda": torch.version.cuda}\n\n\ndef load_tokenizer(config):\n    from transformers import AutoTokenizer\n    tokenizer = AutoTokenizer.from_pretrained(config["model_id"], revision=config["model_revision"], trust_remote_code=False)\n    tokenizer.pad_token = tokenizer.eos_token; tokenizer.padding_side = "right"\n    return tokenizer\n\n\ndef load_training_model(config, adapter_path=None):\n    from peft import LoraConfig, PeftModel, get_peft_model\n    model = load_base(config, training=True)\n    if adapter_path is None:\n        model = get_peft_model(model, LoraConfig(r=config["lora_r"], lora_alpha=config["lora_alpha"],\n            lora_dropout=0.0, target_modules=config.get("target_modules", ["q_proj", "k_proj", "v_proj", "o_proj"]),\n            task_type="CAUSAL_LM", bias="none"))\n    else:\n        # The saved SFT adapter is the one trainable adapter, on the original base.\n        model = PeftModel.from_pretrained(model, adapter_path, is_trainable=True)\n    model.peft_config["default"].revision = config["model_revision"]\n    for name, parameter in model.named_parameters():\n        if parameter.requires_grad:\n            if "lora_" not in name: raise RuntimeError("Only LoRA parameters may be trained")\n            parameter.data = parameter.data.float()\n    for module in model.modules():\n        if isinstance(module, torch.nn.Dropout): module.p = 0.0\n    return model\n\n\ndef load_evaluation_model(config, method, run_dir):\n    from peft import PeftModel\n    model = load_base(config)\n    if method != "base": model = PeftModel.from_pretrained(model, run_dir / "adapters" / method, is_trainable=False)\n    return model.eval()\n\n\ndef validate_config(config):\n    revision = config.get("model_revision", "")\n    if not isinstance(revision, str) or len(revision) != 40 or any(c not in "0123456789abcdef" for c in revision):\n        raise ValueError("Pin a 40-character hexadecimal model commit")\n    for name in ["sft_steps", "dpo_steps", "gradient_accumulation", "save_every", "max_new_tokens", "max_length", "dpo_examples", "bootstrap_resamples"]:\n        if type(config.get(name)) is not int or config[name] < 1: raise ValueError(f"{name} must be positive integer")\n    if config.get("eval_limit") is not None: raise ValueError("Evaluate the complete fixed splits")\n    for name in ["learning_rate_sft", "learning_rate_dpo", "beta"]:\n        if not math.isfinite(config[name]) or config[name] <= 0: raise ValueError(f"Invalid {name}")\n\n\ndef protocol_for(config):\n    return {"methods": ["base", "sft", "dpo"], "reference_baseline": "rule_engine",\n        "selection": "None: fixed steps and unchanged decoding; no development or test tuning",\n        "primary_metric": "structured_pass", "test_evaluation_after_both_training_stages": True,\n        "max_new_tokens": config["max_new_tokens"], "do_sample": False, "adapter_merging": False,\n        "reply_semantic_quality_measured_by_primary_metric": False}\n\n\ndef validate_inputs(run_dir):\n    """Reject edited configuration, data, source, or a changed software/GPU environment."""\n    from support_data import validate_dataset\n    run_dir = Path(run_dir).resolve()\n    manifest_path = run_dir / "inputs_manifest.json"\n    if not manifest_path.is_file():\n        raise ValueError("Run has no staged input seal; legacy recovery must be explicitly audited before continuation")\n    manifest = read_json(manifest_path)\n    required = {"config.json", "protocol.json", "environment.json", "data_manifest.json", "data_audit.json",\n        "code/train_evaluate.py", "code/support_data.py", "data/train.jsonl", "data/development.jsonl", "data/test.jsonl"}\n    if set(manifest) != required: raise ValueError("Input manifest coverage differs")\n    for relative, expected in manifest.items():\n        if digest(run_dir / relative) != expected: raise ValueError(f"Saved input changed: {relative}")\n    for filename in ["train_evaluate.py", "support_data.py"]:\n        if digest(Path(__file__).with_name(filename)) != manifest["code/" + filename]:\n            raise ValueError(f"Runner/source differs from saved experiment: {filename}; restore run/code version")\n    config = read_json(run_dir / "config.json"); validate_config(config)\n    if read_json(run_dir / "protocol.json") != protocol_for(config): raise ValueError("Protocol differs")\n    data = {split: read_jsonl(run_dir / "data" / (split + ".jsonl")) for split in ["train", "development", "test"]}\n    if validate_dataset(data) != read_json(run_dir / "data_audit.json"): raise ValueError("Dataset audit differs")\n    if config["dpo_examples"] > len(data["train"]): raise ValueError("DPO subset exceeds training data")\n    return config, data, read_json(run_dir / "run_status.json")\n\n\ndef validate_environment(run_dir):\n    current = environment_info(); original = read_json(run_dir / "environment.json")\n    if current != original:\n        raise ValueError("Runtime differs from saved experiment (Python/packages/GPU/CUDA); use the recorded environment")\n    return current\n\n\ndef validate_adapter(run_dir, stage):\n    directory = run_dir / "adapters" / stage\n    manifest = read_json(directory / "adapter_manifest.json")\n    actual = {p.name for p in directory.iterdir() if p.is_file() and p.name != "adapter_manifest.json"}\n    if actual != set(manifest) or "adapter_model.safetensors" not in actual: raise ValueError(f"Incomplete {stage} adapter")\n    for filename, expected in manifest.items():\n        if digest(directory / filename) != expected: raise ValueError(f"Changed {stage} adapter: {filename}")\n    info = read_json(run_dir / "adapter_backups.json")[stage]\n    if digest(directory / "adapter_manifest.json") != info["adapter_manifest_sha256"]:\n        raise ValueError(f"{stage} adapter manifest changed")\n    config = read_json(run_dir / "config.json"); adapter_config = read_json(directory / "adapter_config.json")\n    if adapter_config.get("revision") != config["model_revision"] or adapter_config.get("base_model_name_or_path") != config["model_id"]:\n        raise ValueError(f"{stage} adapter base/revision differs")\n    if digest(directory / "adapter_model.safetensors") != info["adapter_weights_sha256"]:\n        raise ValueError(f"{stage} adapter disagrees with completed-stage record")\n    archive = run_dir.parent / info["zip_name"]\n    if not archive.is_file() or digest(archive) != info["zip_sha256"]: raise ValueError(f"Missing or changed {stage} backup ZIP")\n    return info\n\n\ndef validate_saved_run(run_dir, required_stages=("sft",)):\n    """Read-only validation for notebook reattachment; does not load models."""\n    out = Path(run_dir).resolve(); config, data, status = validate_inputs(out)\n    for stage in required_stages:\n        if stage not in ("sft", "dpo") or stage not in status["completed_stages"]:\n            raise ValueError(f"Required completed stage is missing: {stage}")\n        validate_adapter(out, stage)\n        logs = read_jsonl(out / (stage + "_training_log.jsonl"))\n        if [r.get("step") for r in logs] != list(range(1, config[stage + "_steps"] + 1)):\n            raise ValueError(f"Incomplete {stage} training log")\n        timing = read_json(out / "timing.json")[stage]\n        if timing.get("optimizer_steps") != config[stage + "_steps"]:\n            raise ValueError(f"Incomplete {stage} timing")\n    return out\n\n\ndef stage_status(run_dir, status, stage, state, **extra):\n    status.update(status=state, active_stage=stage, benchmark_completed=False, **extra)\n    status.pop("error", None); status.pop("error_type", None); status.pop("failed_stage", None)\n    write_json(run_dir / "run_status.json", status)\n\n\ndef interrupted(run_dir, status, stage, exc):\n    status.update(status="interrupted", active_stage=stage, failed_stage=stage,\n        error_type=type(exc).__name__, error=str(exc), benchmark_completed=False)\n    write_json(run_dir / "run_status.json", status)\n    try: finalize_evidence(run_dir, "partial_evidence")\n    except Exception as export_error: print(f"Partial evidence export failed: {export_error}", flush=True)\n\n\ndef release_gpu():\n    gc.collect()\n    if torch.cuda.is_available(): torch.cuda.empty_cache()\n\n\ndef run_sft(config_path="config.json", output_root="artifacts"):\n    """Start one new experiment, persist the SFT adapter and return before DPO."""\n    from transformers import set_seed\n    from support_data import build_dataset, validate_dataset\n    require_cuda()\n    config = read_json(config_path); validate_config(config)\n    set_seed(config["seed"]); torch.backends.cuda.matmul.allow_tf32 = False\n    output_root = Path(output_root).resolve()\n    if (output_root / "run_status.json").exists(): raise ValueError("output_root is an existing run; do not retrain it")\n    name = "generation_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:6]\n    out = output_root / name; out.mkdir(parents=True)\n    print(f"Persistent run folder: {out}", flush=True)\n    write_json(out / "config.json", config); write_json(out / "environment.json", environment_info())\n    write_json(out / "protocol.json", protocol_for(config))\n    (out / "code").mkdir(); (out / "data").mkdir()\n    for filename in ["train_evaluate.py", "support_data.py"]:\n        shutil.copy2(Path(__file__).with_name(filename), out / "code" / filename)\n    status = {"status": "running", "active_stage": "sft", "completed_stages": [],\n              "benchmark_completed": False, "evaluation_complete": False}\n    write_json(out / "run_status.json", status)\n    write_json(out / "timing.json", {}); write_json(out / "adapter_backups.json", {})\n    model = None\n    try:\n        data = build_dataset(config["seed"])\n        if config["dpo_examples"] > len(data["train"]): raise ValueError("DPO subset exceeds training data")\n        write_json(out / "data_audit.json", validate_dataset(data))\n        for split, rows in data.items(): write_jsonl(out / "data" / (split + ".jsonl"), rows)\n        write_json(out / "data_manifest.json", {s: {"n": len(rows), "sha256": digest(out / "data" / (s + ".jsonl"))} for s, rows in data.items()})\n        sealed = ["config.json", "protocol.json", "environment.json", "data_manifest.json", "data_audit.json",\n            "code/train_evaluate.py", "code/support_data.py", "data/train.jsonl", "data/development.jsonl", "data/test.jsonl"]\n        write_json(out / "inputs_manifest.json", {name: digest(out / name) for name in sealed})\n        tokenizer = load_tokenizer(config)\n        encoded = [(encode_completion(tokenizer, r["prompt"], r["completion"], config["max_length"]),\n                    encode_completion(tokenizer, r["prompt"], r["rejected"], config["max_length"])) for r in data["train"]]\n        for split in ["development", "test"]:\n            for row in data[split]:\n                length = len(tokenizer.apply_chat_template([{"role": "user", "content": row["prompt"]}], tokenize=True, add_generation_prompt=True))\n                if length + config["max_new_tokens"] > config["max_length"]: raise ValueError(f"{split}/{row[\'id\']}: generation budget too long")\n        started = time.perf_counter(); model = load_training_model(config)\n        timings = {"training_model_loading_seconds": time.perf_counter() - started}\n        trainable, total = model.get_nb_trainable_parameters()\n        write_json(out / "parameters.json", {"trainable_adapter_parameters": trainable, "total_parameters": total,\n            "trainable_fraction": trainable / total, "lora_dropout": 0.0, "base_storage": "NF4 with double quantization",\n            "compute_dtype": "float16", "adapter_dtype": "float32"})\n        timings["sft"] = training_loop(model, tokenizer, encoded, data["train"], config, out, "sft")\n        write_json(out / "timing.json", timings)\n        backup = stage_backup(model, tokenizer, out, "sft", None)\n        write_json(out / "adapter_backups.json", {"sft": backup})\n        status["completed_stages"] = ["sft"]\n        stage_status(out, status, "sft", "sft_complete")\n        finalize_evidence(out, "sft_evidence")\n        print(f"SFT saved. Next: run_dpo({str(out)!r})", flush=True)\n        return out\n    except BaseException as exc:\n        interrupted(out, status, "sft", exc); raise\n    finally:\n        del model; release_gpu()\n\n\ndef run_dpo(run_dir):\n    """Continue exactly the saved SFT adapter; never repeat an attempted DPO stage."""\n    from transformers import set_seed\n    out = Path(run_dir).resolve(); config, data, status = validate_inputs(out)\n    if "dpo" in status["completed_stages"]:\n        validate_saved_run(out, ("sft", "dpo"))\n        print(f"DPO already complete; unchanged: {out}", flush=True)\n        return out\n    if status["completed_stages"] != ["sft"]: raise ValueError("A completed SFT stage is required")\n    if (out / "dpo_attempt.json").exists(): raise ValueError("DPO was already attempted; automatic optimizer-state resume is not supported")\n    validate_adapter(out, "sft"); require_cuda(); validate_environment(out)\n    set_seed(config["seed"]); torch.backends.cuda.matmul.allow_tf32 = False\n    tokenizer = load_tokenizer(config)\n    subset = np.random.default_rng(config["seed"] + 2).permutation(len(data["train"]))[:config["dpo_examples"]].tolist()\n    rows = [data["train"][i] for i in subset]\n    encoded = [(encode_completion(tokenizer, r["prompt"], r["completion"], config["max_length"]),\n                encode_completion(tokenizer, r["prompt"], r["rejected"], config["max_length"])) for r in rows]\n    write_json(out / "dpo_training_ids.json", [r["id"] for r in rows])\n    write_json(out / "training_mixture.json", {\n        "sft_categories": dict(Counter(r["category"] for r in data["train"])),\n        "dpo_categories": dict(Counter(r["category"] for r in rows)),\n        "dpo_rejection_types": dict(Counter(r["rejection_type"] for r in rows)),\n        "subset_selection": "Fixed seeded random subset of training rows; no held-out information used"})\n    stage_status(out, status, "reference_scoring", "running")\n    model = None; timings = read_json(out / "timing.json")\n    try:\n        started = time.perf_counter(); model = load_training_model(config, out / "adapters" / "sft")\n        timings["dpo_model_loading_seconds"] = time.perf_counter() - started\n        reference, info = reference_scores(model, encoded, rows, out)\n        write_json(out / "dpo_reference.json", info)\n        write_json(out / "dpo_attempt.json", {"started_at": datetime.now(timezone.utc).isoformat(),\n            "sft_weights_sha256": digest(out / "adapters/sft/adapter_model.safetensors"),\n            "reference_sha256": info["sha256"], "continued_adapter": "saved SFT on original pinned base; no merge"})\n        stage_status(out, status, "dpo", "running")\n        timings["dpo"] = training_loop(model, tokenizer, encoded, rows, config, out, "dpo", reference)\n        write_json(out / "timing.json", timings)\n        if digest(out / "dpo_reference_logps.jsonl") != info["sha256"]: raise ValueError("Frozen reference changed")\n        backup = stage_backup(model, tokenizer, out, "dpo", None)\n        backups = read_json(out / "adapter_backups.json"); backups["dpo"] = backup\n        write_json(out / "adapter_backups.json", backups)\n        status["completed_stages"] = ["sft", "dpo"]\n        stage_status(out, status, "dpo", "training_complete")\n        finalize_evidence(out, "training_evidence")\n        print(f"DPO saved. Next: evaluate_saved({str(out)!r})", flush=True)\n        return out\n    except BaseException as exc:\n        write_json(out / "timing.json", timings)\n        interrupted(out, status, status.get("active_stage", "dpo"), exc); raise\n    finally:\n        del model; release_gpu()\n\n\ndef record_hash(record):\n    return hashlib.sha256(json.dumps({k: v for k, v in record.items() if k != "record_sha256"},\n        sort_keys=True, separators=(",", ":"), allow_nan=False).encode()).hexdigest()\n\n\ndef evaluation_plan(data):\n    return [(method, split, row) for method in ["base", "sft", "dpo", "rule_engine"]\n            for split in ["development", "test"] for row in data[split]]\n\n\ndef validate_predictions(records, plan, config, provenance):\n    from support_data import score_response, rule_response\n    if len(records) > len(plan): raise ValueError("Too many predictions")\n    for record, (method, split, row) in zip(records, plan):\n        if (record.get("method"), record.get("split"), record.get("id")) != (method, split, row["id"]):\n            raise ValueError("Saved predictions are not the canonical uninterrupted prefix")\n        if record.get("provenance_sha256") != provenance or record.get("record_sha256") != record_hash(record):\n            raise ValueError("Saved prediction contents/provenance changed")\n        response = record.get("response")\n        if not isinstance(response, str) or record.get("category") != row["category"]:\n            raise ValueError("Malformed prediction")\n        if record.get("scores") != score_response(row, response) or type(record.get("hit_token_limit")) is not bool:\n            raise ValueError("Prediction scores or limit flag differ")\n        tokens = record.get("generated_tokens_including_eos"); seconds = record.get("generation_seconds")\n        if method == "rule_engine":\n            if response != rule_response(row) or tokens is not None or seconds is not None or record["hit_token_limit"]:\n                raise ValueError("Rule baseline differs")\n        elif type(tokens) is not int or not 1 <= tokens <= config["max_new_tokens"] or type(seconds) not in (float, int) or not math.isfinite(seconds) or seconds < 0:\n            raise ValueError("Invalid generation timing/token count")\n        elif record["hit_token_limit"] and tokens != config["max_new_tokens"]:\n            raise ValueError("Invalid token-limit count")\n\n\ndef export_evaluation(out, config, data, predictions):\n    from support_data import summarize\n    indexed = {(p["method"], p["split"], p["id"]): p for p in predictions}\n    metrics = {method: {split: summarize(data[split], [indexed[method, split, r["id"]]["response"] for r in data[split]])\n        for split in ["development", "test"]} for method in ["base", "sft", "dpo", "rule_engine"]}\n    write_json(out / "metrics.json", metrics)\n    write_json(out / "paired_comparisons.json", paired_comparisons(predictions, config["seed"], config["bootstrap_resamples"]))\n    import io\n    buffer = io.StringIO(newline="")\n    fields = ["method", "split"] + [k for k, v in metrics["base"]["test"].items() if not isinstance(v, (dict, list))]\n    writer = csv.DictWriter(buffer, fieldnames=fields); writer.writeheader()\n    for method, splits in metrics.items():\n        for split, values in splits.items(): writer.writerow({"method": method, "split": split, **{k: values[k] for k in fields[2:]}})\n    atomic_text(out / "metrics_summary.csv", buffer.getvalue())\n    rng = random.Random(config["seed"]); ids = set()\n    for category in sorted({r["category"] for r in data["test"]}):\n        ids.update(rng.sample(sorted(r["id"] for r in data["test"] if r["category"] == category), 2))\n    review = [p for p in predictions if p["split"] == "test" and p["method"] != "rule_engine" and p["id"] in ids]\n    rng.shuffle(review); opaque = rng.sample(range(10**8, 10**9), len(review))\n    mapping = [{"review_id": f"R{opaque[i]}", "method": p["method"], "id": p["id"]} for i, p in enumerate(review)]\n    write_json(out / "review_key.json", {"instructions": "Open only after recording blinded ratings; method identity is hidden in the review CSV, not cryptographically protected.", "mapping": mapping})\n    row_map = {r["id"]: r for r in data["test"]}; buffer = io.StringIO(newline="")\n    writer = csv.DictWriter(buffer, fieldnames=["review_id", "prompt", "response", "reply_factual", "instruction_following", "unsupported_advice", "notes"])\n    writer.writeheader()\n    for i, p in enumerate(review): writer.writerow({"review_id": mapping[i]["review_id"], "prompt": row_map[p["id"]]["prompt"], "response": p["response"]})\n    atomic_text(out / "manual_review.csv", buffer.getvalue())\n\n\ndef evaluate_saved(run_dir):\n    """Evaluate saved stages, checkpoint every ten responses, and safely retry gaps."""\n    from support_data import rule_response, score_response\n    from transformers import set_seed\n    out = Path(run_dir).resolve(); config, data, status = validate_inputs(out)\n    if status["completed_stages"] != ["sft", "dpo"]: raise ValueError("Both saved training stages are required")\n    adapter_info = {stage: validate_adapter(out, stage) for stage in ["sft", "dpo"]}\n    receipt_path = out.parent / (out.name + "_completion.json")\n    if receipt_path.exists():\n        receipt = read_json(receipt_path)\n        archive = out.parent / (out.name + "_results.zip")\n        if status.get("status") != "complete" or digest(out / "manifest_sha256.json") != receipt["manifest_sha256"] or not archive.is_file() or digest(archive) != receipt["results_zip_sha256"]:\n            raise ValueError("Finalized evidence or its archive changed; refusing to overwrite it")\n        from scripts.verify_results import verify\n        verify(out)\n        print(f"Already complete; unchanged: {out}", flush=True)\n        return out\n    # The receipt is the final commit. Without it, an interrupted ZIP export can\n    # be completed using validated saved predictions, without loading a model.\n    environment = read_json(out / "environment.json")\n    provenance = {"inputs_manifest_sha256": digest(out / "inputs_manifest.json"),\n        "adapter_weights_sha256": {s: v["adapter_weights_sha256"] for s, v in adapter_info.items()},\n        "decoding": {"do_sample": False, "max_new_tokens": config["max_new_tokens"], "max_length": config["max_length"],\n            "use_cache": True, "batch_size": 1, "generated_continuation_only": True, "allow_tf32": False},\n        "environment": environment, "case_order": "base/sft/dpo/rule_engine; development/test; saved data order"}\n    path = out / "evaluation_manifest.json"\n    if path.exists() and read_json(path) != provenance: raise ValueError("Evaluation settings/provenance changed")\n    if not path.exists(): write_json(path, provenance)\n    provenance_hash = digest(path); plan = evaluation_plan(data)\n    predictions = read_jsonl(out / "predictions.jsonl") if (out / "predictions.jsonl").exists() else []\n    validate_predictions(predictions, plan, config, provenance_hash)\n    if len(predictions) < len(plan):\n        require_cuda(); validate_environment(out)\n        set_seed(config["seed"]); torch.backends.cuda.matmul.allow_tf32 = False\n        tokenizer = load_tokenizer(config)\n    model = None; current_method = None\n    timings = read_json(out / "timing.json")\n    sessions_path = out / "evaluation_sessions.json"\n    sessions = read_json(sessions_path) if sessions_path.exists() else []\n    for previous in sessions:\n        if previous.get("status") == "running":\n            previous.update(status="interrupted", error_type="UncleanSessionEnd", ending_predictions=len(predictions))\n    session = {"started_at": datetime.now(timezone.utc).isoformat(), "starting_predictions": len(predictions), "status": "running"}\n    sessions.append(session); write_json(sessions_path, sessions)\n    stage_status(out, status, "evaluation", "running", evaluation_complete=False)\n    print(f"Evaluation: {len(predictions)}/{len(plan)} saved; resuming missing responses only.", flush=True)\n    try:\n        for method, split, row in plan[len(predictions):]:\n            if method != current_method:\n                if model is not None: del model; model = None; release_gpu()\n                current_method = method\n                if method != "rule_engine":\n                    started = time.perf_counter(); model = load_evaluation_model(config, method, out)\n                    key = method + "_evaluation_loading_seconds"\n                    timings[key] = timings.get(key, 0.0) + time.perf_counter() - started\n                    torch.cuda.reset_peak_memory_stats()\n                    write_json(out / "timing.json", timings)\n                print(f"Evaluating {method}: {split}", flush=True)\n            if method == "rule_engine":\n                text, timing = rule_response(row), {"generation_seconds": None, "generated_tokens_including_eos": None, "hit_token_limit": False}\n            else:\n                text, timing = generate(model, tokenizer, row, config)\n                key = method + "_evaluation_peak_allocated_gpu_bytes"\n                timings[key] = max(timings.get(key, 0), torch.cuda.max_memory_allocated())\n            record = {"method": method, "split": split, "id": row["id"], "category": row["category"],\n                "response": text, "scores": score_response(row, text), **timing, "provenance_sha256": provenance_hash}\n            record["record_sha256"] = record_hash(record); predictions.append(record)\n            if len(predictions) % 10 == 0 or len(predictions) == len(plan):\n                write_jsonl(out / "predictions.jsonl", predictions); write_json(out / "timing.json", timings)\n                print(f"Evaluation saved: {len(predictions)}/{len(plan)} ({method}/{split})", flush=True)\n        validate_predictions(predictions, plan, config, provenance_hash)\n        export_evaluation(out, config, data, predictions)\n        timings["scope"] = "Single-run wall times; generation includes GPU synchronization; recovery loading times accumulated; no production cost measured"\n        write_json(out / "timing.json", timings)\n        session.update(status="complete", ending_predictions=len(predictions)); write_json(sessions_path, sessions)\n        status.update(status="complete", active_stage="evaluation", benchmark_completed=True, evaluation_complete=True, human_review_completed=False)\n        write_json(out / "run_status.json", status)\n        archive = finalize_evidence(out)\n        write_json(receipt_path, {"manifest_sha256": digest(out / "manifest_sha256.json"),\n            "results_zip_sha256": digest(archive), "status": "complete"})\n        return out\n    except BaseException as exc:\n        write_jsonl(out / "predictions.jsonl", predictions); write_json(out / "timing.json", timings)\n        session.update(status="interrupted", ending_predictions=len(predictions), error_type=type(exc).__name__)\n        write_json(sessions_path, sessions); interrupted(out, status, "evaluation", exc); raise\n    finally:\n        del model; release_gpu()\n\n\ndef run(config_path="config.json", output_root="artifacts", stage_callback=None):\n    """Legacy all-stage wrapper. Prefer separate stage calls for recoverable notebooks."""\n    out = run_sft(config_path, output_root)\n    if stage_callback is not None: stage_callback("sft", out, out.parent / read_json(out / "adapter_backups.json")["sft"]["zip_name"])\n    run_dpo(out)\n    if stage_callback is not None: stage_callback("dpo", out, out.parent / read_json(out / "adapter_backups.json")["dpo"]["zip_name"])\n    return evaluate_saved(out)\n\ndef smoke():\n    """Execute the integration test against random weights; never download a model."""\n    import unittest\n    suite = unittest.defaultTestLoader.discover(str(Path(__file__).parent / "tests"), pattern="test_training.py")\n    result = unittest.TextTestRunner(verbosity=2).run(suite)\n    if not result.wasSuccessful() or result.skipped:\n        raise RuntimeError("Training smoke did not complete every test; install the required packages")\n    return {"status": "passed", "tests_run": result.testsRun, "model": "tiny randomly initialized Llama",\n            "scope": "CPU mechanics only; not TinyLlama, GPU quantization, benchmark quality, or runtime validation"}\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--config", default=str(Path(__file__).with_name("config.json")))\n    parser.add_argument("--output-root", default="artifacts")\n    parser.add_argument("--smoke", action="store_true")\n    args = parser.parse_args()\n    print(json.dumps(smoke(), indent=2) if args.smoke else str(run(args.config, args.output_root)))\n', 'config.json': '{\n  "experiment": "support_response_sft_dpo",\n  "model_id": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",\n  "model_revision": "fe8a4ea1ffedaf415f4da2f062534de366a451e6",\n  "seed": 2026,\n  "max_length": 512,\n  "max_new_tokens": 128,\n  "lora_r": 16,\n  "lora_alpha": 32,\n  "lora_dropout": 0.0,\n  "sft_steps": 120,\n  "dpo_steps": 60,\n  "dpo_examples": 240,\n  "gradient_accumulation": 4,\n  "learning_rate_sft": 0.0002,\n  "learning_rate_dpo": 5e-05,\n  "beta": 0.1,\n  "save_every": 30,\n  "eval_limit": null,\n  "bootstrap_resamples": 2000\n}\n', 'requirements.txt': '# Python 3.10+; keep the CUDA-enabled PyTorch supplied by a fresh Colab GPU runtime.\ntorch>=2.6,<3\ntransformers==4.57.6\npeft==0.18.1\naccelerate==1.12.0\nbitsandbytes==0.48.2\nnumpy==2.3.5\n', 'scripts/verify_results.py': '"""Verify exported evidence without importing torch or loading any model weights.\n\nThis checks internal consistency, not that a GPU run really occurred. Keep the\noriginal exported folder unchanged; enter human ratings in a separate copy.\n"""\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport math\nimport random\nimport sys\nfrom collections import Counter\nfrom pathlib import Path\n\nimport numpy as np\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom support_data import CATEGORIES, build_dataset, rule_response, score_response, summarize, validate_dataset\n\nMETHODS = ("base", "sft", "dpo", "rule_engine")\nSPLITS = ("development", "test")\nEXCLUDED = {"adapters", "checkpoints"}\n\n\nclass VerificationError(ValueError):\n    pass\n\n\ndef require(condition, message):\n    if not condition:\n        raise VerificationError(message)\n\n\ndef digest(path):\n    return hashlib.sha256(path.read_bytes()).hexdigest()\n\n\ndef strict_object(pairs):\n    result = {}\n    for key, value in pairs:\n        require(key not in result, f"Duplicate JSON key: {key}")\n        result[key] = value\n    return result\n\n\ndef loads(text):\n    return json.loads(text, object_pairs_hook=strict_object,\n                      parse_constant=lambda x: (_ for _ in ()).throw(VerificationError(f"Nonfinite JSON value: {x}")))\n\n\ndef read_json(path):\n    return loads(path.read_text(encoding="utf-8"))\n\n\ndef read_jsonl(path):\n    return [loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]\n\n\ndef read_csv(path):\n    with path.open(newline="", encoding="utf-8") as stream:\n        reader = csv.DictReader(stream)\n        fields = reader.fieldnames or []\n        require(len(fields) == len(set(fields)), f"Duplicate CSV columns: {path.name}")\n        rows = list(reader)\n    require(all(None not in row and all(value is not None for value in row.values()) for row in rows),\n            f"Malformed CSV row: {path.name}")\n    return fields, rows\n\n\ndef finite_number(value, label, nonnegative=False):\n    require(type(value) in (int, float) and math.isfinite(value), f"{label} must be finite numeric")\n    require(not nonnegative or value >= 0, f"{label} must be nonnegative")\n\n\ndef same(actual, expected, label):\n    """Compare nested metrics while keeping boolean flags distinct from 0/1."""\n    if isinstance(expected, dict):\n        require(isinstance(actual, dict) and actual.keys() == expected.keys(), f"{label}: keys differ")\n        for key in expected:\n            same(actual[key], expected[key], f"{label}.{key}")\n    elif isinstance(expected, list):\n        require(isinstance(actual, list) and len(actual) == len(expected), f"{label}: list length differs")\n        for i, (a, b) in enumerate(zip(actual, expected)):\n            same(a, b, f"{label}[{i}]")\n    elif type(expected) is bool:\n        require(type(actual) is bool and actual == expected, f"{label}: boolean differs")\n    elif type(expected) in (int, float):\n        finite_number(actual, label)\n        require(math.isclose(actual, expected, rel_tol=1e-12, abs_tol=1e-12), f"{label}: number differs")\n    else:\n        require(actual == expected, f"{label}: value differs")\n\n\ndef expected_order(ids, steps, accumulation, seed):\n    rng = np.random.default_rng(seed)\n    result = []\n    while len(result) < steps * accumulation:\n        result.extend(ids[i] for i in rng.permutation(len(ids)))\n    return result[:steps * accumulation]\n\n\ndef recompute_comparisons(predictions, seed, resamples):\n    indexed = {(p["method"], p["id"]): int(p["scores"]["structured_pass"])\n               for p in predictions if p["split"] == "test"}\n    ids = sorted(p["id"] for p in predictions if p["split"] == "test" and p["method"] == "base")\n    result = {}\n    for a, b in (("base", "sft"), ("sft", "dpo"), ("base", "dpo")):\n        differences = [indexed[b, i] - indexed[a, i] for i in ids]\n        frequencies = [differences.count(value) for value in (-1, 0, 1)]\n        bootstrap = np.random.default_rng(seed).multinomial(len(ids), np.asarray(frequencies) / len(ids), size=resamples)\n        bootstrap_means = (bootstrap[:, 2] - bootstrap[:, 0]) / len(ids)\n        result[b + "_minus_" + a] = {\n            "mean_delta": sum(differences) / len(ids),\n            "ci95": np.quantile(bootstrap_means, [0.025, 0.975]).tolist(),\n            "b_better": frequencies[2], "a_better": frequencies[0], "ties": frequencies[1],\n            "resamples": resamples, "seed": seed,\n            "interpretation": "Descriptive paired case bootstrap; no multiplicity or domain-shift adjustment",\n        }\n    return result\n\n\ndef verify(run_dir):\n    root = Path(run_dir).resolve()\n    require(root.is_dir(), "Run folder does not exist")\n    paths = [p for p in root.rglob("*") if p.relative_to(root).parts[0] not in EXCLUDED]\n    require(not any(p.is_symlink() for p in paths), "Evidence must not contain symlinks")\n    files = {p.relative_to(root).as_posix(): p for p in paths if p.is_file() and p.name != "manifest_sha256.json"}\n    manifest = read_json(root / "manifest_sha256.json")\n    require(isinstance(manifest, dict) and set(manifest) == set(files), "Manifest file coverage differs")\n    for name, path in files.items():\n        require(manifest[name] == digest(path), f"Manifest hash mismatch: {name}")\n    required = {"config.json", "run_status.json", "protocol.json", "environment.json", "parameters.json",\n        "data_audit.json", "data_manifest.json", "predictions.jsonl", "metrics.json", "metrics_summary.csv",\n        "paired_comparisons.json", "dpo_training_ids.json", "dpo_reference.json", "dpo_reference_logps.jsonl",\n        "sft_training_log.jsonl", "dpo_training_log.jsonl", "sft_training_order.json", "dpo_training_order.json",\n        "timing.json", "adapter_backups.json", "manual_review.csv", "review_key.json", "training_mixture.json",\n        "code/train_evaluate.py", "code/support_data.py"}\n    require(required <= files.keys(), f"Required evidence missing: {sorted(required - files.keys())}")\n    config = read_json(root / "config.json")\n    for key in ("seed", "sft_steps", "dpo_steps", "gradient_accumulation", "dpo_examples", "bootstrap_resamples", "max_new_tokens"):\n        require(type(config.get(key)) is int and config[key] >= (0 if key == "seed" else 1), f"Invalid config: {key}")\n    require(config.get("eval_limit") is None, "Limited evaluation is not a completed benchmark")\n    status = read_json(root / "run_status.json")\n    require(status.get("status") == "complete" and status.get("benchmark_completed") is True,\n            "Run is not a completed benchmark")\n    require(status.get("completed_stages") == ["sft", "dpo"], "Training stages are incomplete or reordered")\n    protocol = read_json(root / "protocol.json")\n    same(protocol, {"methods": ["base", "sft", "dpo"], "reference_baseline": "rule_engine",\n        "selection": "None: fixed steps and unchanged decoding; no development or test tuning",\n        "primary_metric": "structured_pass", "test_evaluation_after_both_training_stages": True,\n        "max_new_tokens": config["max_new_tokens"], "do_sample": False, "adapter_merging": False,\n        "reply_semantic_quality_measured_by_primary_metric": False}, "protocol")\n\n    data = {split: read_jsonl(root / "data" / (split + ".jsonl")) for split in ("train", *SPLITS)}\n    audit = validate_dataset(data)\n    require(data == build_dataset(config["seed"]), "Dataset differs from deterministic rebuild")\n    same(read_json(root / "data_audit.json"), audit, "data_audit")\n    same(read_json(root / "data_manifest.json"), {\n        split: {"n": len(rows), "sha256": digest(root / "data" / (split + ".jsonl"))} for split, rows in data.items()\n    }, "data_manifest")\n    rows_by_key = {(split, row["id"]): row for split, rows in data.items() for row in rows}\n\n    predictions = read_jsonl(root / "predictions.jsonl")\n    expected_keys = {(method, split, row["id"]) for method in METHODS for split in SPLITS for row in data[split]}\n    indexed = {}\n    for record in predictions:\n        key = (record.get("method"), record.get("split"), record.get("id"))\n        require(key in expected_keys and key not in indexed, f"Unknown or duplicate prediction: {key}")\n        indexed[key] = record\n        row = rows_by_key[key[1:]]\n        require(record.get("category") == row["category"], f"Prediction category differs: {key}")\n        require(isinstance(record.get("response"), str), f"Response must be text: {key}")\n        same(record.get("scores"), score_response(row, record["response"]), f"prediction scores {key}")\n        require(type(record.get("hit_token_limit")) is bool, f"Invalid token-limit flag: {key}")\n        if key[0] == "rule_engine":\n            require(record["response"] == rule_response(row), f"Rule baseline differs: {key}")\n            require(record.get("generation_seconds") is None and record.get("generated_tokens_including_eos") is None\n                    and record["hit_token_limit"] is False, f"Rule timing differs: {key}")\n        else:\n            finite_number(record.get("generation_seconds"), f"generation time {key}", True)\n            tokens = record.get("generated_tokens_including_eos")\n            require(type(tokens) is int and 1 <= tokens <= config["max_new_tokens"], f"Invalid token count: {key}")\n            require(not record["hit_token_limit"] or tokens == config["max_new_tokens"], f"Token-limit count differs: {key}")\n    require(indexed.keys() == expected_keys, "Prediction coverage is incomplete")\n    recomputed = {method: {split: summarize(data[split], [indexed[method, split, row["id"]]["response"]\n        for row in data[split]]) for split in SPLITS} for method in METHODS}\n    same(read_json(root / "metrics.json"), recomputed, "metrics")\n    scalar_fields = [k for k, value in recomputed["base"]["test"].items() if not isinstance(value, (dict, list))]\n    fields, csv_rows = read_csv(root / "metrics_summary.csv")\n    require(fields == ["method", "split", *scalar_fields], "Metric CSV columns differ")\n    seen = set()\n    for row in csv_rows:\n        key = (row["method"], row["split"])\n        require(key[0] in METHODS and key[1] in SPLITS and key not in seen, "Unknown/duplicate metric CSV row")\n        seen.add(key)\n        for field in scalar_fields:\n            same(float(row[field]), recomputed[key[0]][key[1]][field], f"metric CSV {key}.{field}")\n    require(seen == {(method, split) for method in METHODS for split in SPLITS}, "Metric CSV coverage differs")\n    same(read_json(root / "paired_comparisons.json"), recompute_comparisons(predictions, config["seed"],\n         config["bootstrap_resamples"]), "paired_comparisons")\n\n    train_ids = [r["id"] for r in data["train"]]\n    require(config["dpo_examples"] <= len(train_ids), "DPO subset exceeds training data")\n    dpo_ids = [train_ids[i] for i in np.random.default_rng(config["seed"] + 2).permutation(len(train_ids))[:config["dpo_examples"]]]\n    same(read_json(root / "dpo_training_ids.json"), dpo_ids, "dpo_training_ids")\n    dpo_rows = [rows_by_key["train", ticket] for ticket in dpo_ids]\n    same(read_json(root / "training_mixture.json"), {\n        "sft_categories": dict(Counter(row["category"] for row in data["train"])),\n        "dpo_categories": dict(Counter(row["category"] for row in dpo_rows)),\n        "dpo_rejection_types": dict(Counter(row["rejection_type"] for row in dpo_rows)),\n        "subset_selection": "Fixed seeded random subset of training rows; no held-out information used"}, "training_mixture")\n    refs = read_jsonl(root / "dpo_reference_logps.jsonl")\n    require([record.get("id") for record in refs] == dpo_ids, "DPO references differ from training subset")\n    for record in refs:\n        require(set(record) == {"id", "chosen_logp", "rejected_logp"}, "DPO reference fields differ")\n        for field in ("chosen_logp", "rejected_logp"):\n            finite_number(record[field], field)\n            require(record[field] <= 0, "Completion log probability cannot be positive")\n    ref_info = read_json(root / "dpo_reference.json")\n    require(ref_info.get("sha256") == digest(root / "dpo_reference_logps.jsonl"), "DPO reference hash differs")\n    require(ref_info.get("reference") == "Frozen SFT adapter on the same NF4 base"\n            and ref_info.get("reduction") == "completion token sum including EOS", "DPO reference description differs")\n    finite_number(ref_info.get("seconds"), "reference seconds", True)\n    timings = read_json(root / "timing.json")\n    for stage, ids, seed in (("sft", train_ids, config["seed"]), ("dpo", dpo_ids, config["seed"] + 1)):\n        steps = config[stage + "_steps"]\n        logs = read_jsonl(root / (stage + "_training_log.jsonl"))\n        require([row.get("step") for row in logs] == list(range(1, steps + 1)), f"{stage} log steps differ")\n        for row in logs:\n            for field in ("loss", "gradient_norm_before_clip"):\n                finite_number(row.get(field), f"{stage} {field}", True)\n        same(read_json(root / (stage + "_training_order.json")),\n             expected_order(ids, steps, config["gradient_accumulation"], seed), stage + "_training_order")\n        require(timings[stage].get("optimizer_steps") == steps\n                and timings[stage].get("training_example_presentations") == steps * config["gradient_accumulation"],\n                f"{stage} timing counts differ")\n        for field in ("seconds", "peak_allocated_gpu_bytes", "peak_reserved_gpu_bytes"):\n            finite_number(timings[stage].get(field), stage + " " + field, True)\n\n    review_rng = random.Random(config["seed"])\n    review_ids = set()\n    for category in sorted(CATEGORIES):\n        candidates = sorted(row["id"] for row in data["test"] if row["category"] == category)\n        review_ids.update(review_rng.sample(candidates, 2))\n    expected_review = {(method, ticket) for method in METHODS[:3] for ticket in review_ids}\n    mapping = read_json(root / "review_key.json")["mapping"]\n    require(len(mapping) == 36 and {(row.get("method"), row.get("id")) for row in mapping} == expected_review,\n            "Review key must cover the fixed balanced 12 tickets and three models")\n    by_review = {row["review_id"]: row for row in mapping}\n    require(len(by_review) == 36 and all(isinstance(key, str) and key.startswith("R") and key[1:].isdigit()\n            for key in by_review), "Review IDs are missing, duplicated, or malformed")\n    fields, reviews = read_csv(root / "manual_review.csv")\n    require(fields == ["review_id", "prompt", "response", "reply_factual", "instruction_following", "unsupported_advice", "notes"],\n            "Review CSV columns differ or reveal method names")\n    require(len(reviews) == 36 and Counter(row["review_id"] for row in reviews) == Counter(by_review.keys()),\n            "Review CSV coverage differs")\n    for row in reviews:\n        identity = by_review[row["review_id"]]\n        require(row["prompt"] == rows_by_key["test", identity["id"]]["prompt"], "Review prompt differs")\n        require(row["response"] == indexed[identity["method"], "test", identity["id"]]["response"], "Review response differs")\n    rated = sum(all(row[k].strip() for k in ("reply_factual", "instruction_following", "unsupported_advice")) for row in reviews)\n    return {"status": "verified", "evidence_files": len(files), "predictions": len(predictions),\n            "test_examples_per_method": len(data["test"]), "review_rows": len(reviews), "review_rows_rated": rated,\n            "scope": "Internal evidence consistency only; does not authenticate GPU execution or validate reply factuality."}\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("run_folder", type=Path)\n    args = parser.parse_args()\n    try:\n        report = verify(args.run_folder)\n    except (VerificationError, ValueError, KeyError, TypeError, OSError) as exc:\n        print(json.dumps({"status": "failed", "error": str(exc)}, indent=2))\n        return 1\n    print(json.dumps(report, indent=2))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'tests/test_data.py': 'import json,unittest\nfrom support_data import build_dataset,rule_decision,score_response,validate_dataset\n\nclass PolicyTests(unittest.TestCase):\n    @classmethod\n    def setUpClass(cls): cls.data=build_dataset()\n    def test_boundary_and_precedence(self):\n        f=dict(return_window_days=30,elapsed_days=30,receipt=True,unused=True,damaged=False)\n        self.assertEqual(rule_decision(f),(\'approve\',\'eligible\'))\n        self.assertEqual(rule_decision(dict(f,elapsed_days=31)),(\'deny\',\'outside_window\'))\n        self.assertEqual(rule_decision(dict(f,unused=False)),(\'deny\',\'used_item\'))\n        self.assertEqual(rule_decision(dict(f,elapsed_days=31,receipt=False)),(\'review\',\'missing_receipt\'))\n        self.assertEqual(rule_decision(dict(f,elapsed_days=31,receipt=False,damaged=True)),(\'review\',\'damaged_item\'))\n    def test_groups_and_balances(self):\n        self.assertEqual(validate_dataset(self.data)[\'unique_fact_groups\'],660)\n        self.assertEqual([len(x) for x in self.data.values()],[480,60,120])\n    def test_truth_passes_but_not_every_rejected_fails_fields(self):\n        for row in self.data[\'test\']:\n            self.assertTrue(score_response(row,row[\'completion\'])[\'structured_pass\'])\n            if row[\'rejection_type\']==\'unsupported_reply\':\n                self.assertTrue(score_response(row,row[\'rejected\'])[\'structured_pass\'])\n                self.assertFalse(score_response(row,row[\'rejected\'])[\'reference_reply_exact\'])\n    def test_strict_json(self):\n        r=self.data[\'test\'][0]\n        for text in [r[\'completion\']+\' prose\',\'```\'+r[\'completion\']+\'```\',\'{"reply":NaN}\',\n                     \'{"decision":"approve","decision":"deny"}\']:\n            self.assertFalse(score_response(r,text)[\'json_valid\'])\n        for value in [[],{\'ticket_id\':r[\'id\']},dict(json.loads(r[\'completion\']),extra=True),\n                      dict(json.loads(r[\'completion\']),reply=17),dict(json.loads(r[\'completion\']),reply=\'word \'*61)]:\n            self.assertFalse(score_response(r,json.dumps(value))[\'schema_valid\'])\n    def test_duplicate_fact_group_rejected(self):\n        import copy\n        d=copy.deepcopy(self.data);d[\'test\'][0][\'facts\']=d[\'train\'][0][\'facts\']\n        with self.assertRaises(ValueError): validate_dataset(d)\n\nif __name__==\'__main__\': unittest.main()\n', 'tests/test_training.py': '"""Training-mechanics checks; no model download or benchmark inference."""\nimport importlib.util\nimport math\nimport sys\nimport tempfile\nimport unittest\nfrom pathlib import Path\n\nimport torch\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1]))\nfrom train_evaluate import completion_logps, dpo_loss, encode_completion\n\n\nclass FakeTokenizer:\n    eos_token_id = 9\n\n    def __init__(self, completion_ids=None):\n        self.completion_ids = [4, 5] if completion_ids is None else completion_ids\n        self.chat_calls = []\n        self.token_calls = []\n\n    def apply_chat_template(self, messages, *, tokenize, add_generation_prompt):\n        self.chat_calls.append((messages, tokenize, add_generation_prompt))\n        return [1, 2, 3]\n\n    def __call__(self, text, *, add_special_tokens):\n        self.token_calls.append((text, add_special_tokens))\n        return {"input_ids": self.completion_ids.copy()}\n\n\nclass CompletionEncodingTests(unittest.TestCase):\n    def test_chat_prefix_and_completion_only_labels(self):\n        tokenizer = FakeTokenizer()\n        encoded = encode_completion(tokenizer, "Policy and ticket", \'{"decision":"yes"}\', 6)\n        self.assertEqual(encoded["input_ids"], [1, 2, 3, 4, 5, 9])\n        self.assertEqual(encoded["attention_mask"], [1] * 6)\n        self.assertEqual(encoded["labels"], [-100, -100, -100, 4, 5, 9])\n        self.assertEqual(tokenizer.chat_calls, [\n            ([{"role": "user", "content": "Policy and ticket"}], True, True)])\n        self.assertEqual(tokenizer.token_calls, [(\'{"decision":"yes"}\', False)])\n\n    def test_existing_eos_is_not_duplicated(self):\n        encoded = encode_completion(FakeTokenizer([4, 5, 9]), "prompt", "answer", 6)\n        self.assertEqual(encoded["input_ids"], [1, 2, 3, 4, 5, 9])\n        self.assertEqual(encoded["labels"][-1], 9)\n\n    def test_overlong_example_fails_instead_of_truncating(self):\n        with self.assertRaises(ValueError):\n            encode_completion(FakeTokenizer(), "prompt", "answer", 5)\n\n\nclass CompletionLogProbabilityTests(unittest.TestCase):\n    def setUp(self):\n        self.logits = torch.tensor([[\n            [0.2, -0.1, 0.3, 1.1],\n            [1.2, 0.3, -0.5, 0.7],\n            [-0.3, 1.4, 0.1, 0.0],\n            [0.5, 0.2, 0.1, 0.3],\n            [0.1, 0.5, 0.4, 0.8],\n        ]], dtype=torch.float64)\n        self.labels = torch.tensor([[-100, -100, 2, 1, -100]])\n\n    def test_next_token_shift_and_sum_not_mean(self):\n        expected = sum([\n            -0.5 - math.log(sum(math.exp(x) for x in [1.2, 0.3, -0.5, 0.7])),\n            1.4 - math.log(sum(math.exp(x) for x in [-0.3, 1.4, 0.1, 0.0])),\n        ])\n        actual = completion_logps(self.logits, self.labels)\n        self.assertEqual(tuple(actual.shape), (1,))\n        self.assertAlmostEqual(actual.item(), expected, places=6)\n        self.assertNotAlmostEqual(actual.item(), expected / 2, places=3)\n\n    def test_prompt_and_padding_scores_are_ignored(self):\n        before = completion_logps(self.logits, self.labels)\n        changed = self.logits.clone()\n        changed[:, [0, 3, 4], :] = changed.new_tensor([100.0, -100.0, 0.0, 1.0])\n        torch.testing.assert_close(completion_logps(changed, self.labels), before)\n\n    def test_gradient_reaches_only_scored_prediction_positions(self):\n        logits = self.logits.clone().requires_grad_()\n        completion_logps(logits, self.labels).sum().backward()\n        self.assertTrue(torch.equal(logits.grad[:, [0, 3, 4]], torch.zeros_like(logits.grad[:, [0, 3, 4]])))\n        self.assertGreater(logits.grad[:, [1, 2]].abs().sum().item(), 0)\n\n    def test_independent_batch_rows_and_zero_scored_tokens(self):\n        logits = self.logits.repeat(2, 1, 1)\n        labels = self.labels.repeat(2, 1)\n        labels[1] = -100\n        result = completion_logps(logits, labels)\n        self.assertAlmostEqual(result[0].item(), completion_logps(self.logits, self.labels)[0].item())\n        self.assertEqual(result[1].item(), 0.0)\n\n\nclass DPOLossTests(unittest.TestCase):\n    def test_reference_policy_has_log_two_loss(self):\n        chosen = torch.tensor([-7.0, -10.0])\n        rejected = torch.tensor([-8.0, -12.0])\n        loss = dpo_loss(chosen, rejected, chosen.clone(), rejected.clone(), 0.1)\n        self.assertAlmostEqual(loss.item(), math.log(2), places=6)\n\n    def test_gradient_favors_chosen_over_rejected(self):\n        chosen = torch.tensor([-7.0], requires_grad=True)\n        rejected = torch.tensor([-8.0], requires_grad=True)\n        loss = dpo_loss(chosen, rejected, torch.tensor([-7.0]), torch.tensor([-8.0]), 0.2)\n        loss.backward()\n        self.assertLess(chosen.grad.item(), 0)\n        self.assertGreater(rejected.grad.item(), 0)\n        self.assertAlmostEqual(chosen.grad.item(), -0.1, places=6)\n        self.assertAlmostEqual(rejected.grad.item(), 0.1, places=6)\n\n    def test_reference_adjustment_and_batch_mean(self):\n        policy_chosen = torch.tensor([-4.0, -8.0], dtype=torch.float64)\n        policy_rejected = torch.tensor([-6.0, -7.0], dtype=torch.float64)\n        ref_chosen = torch.tensor([-5.0, -8.0], dtype=torch.float64)\n        ref_rejected = torch.tensor([-6.0, -8.0], dtype=torch.float64)\n        expected = (math.log1p(math.exp(-0.3)) + math.log1p(math.exp(0.3))) / 2\n        actual = dpo_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected, 0.3)\n        self.assertAlmostEqual(actual.item(), expected, places=6)\n\n    def test_invalid_beta_is_rejected(self):\n        x = torch.tensor([-1.0])\n        for beta in [0, -0.1, float("nan"), float("inf")]:\n            with self.subTest(beta=beta), self.assertRaises(ValueError):\n                dpo_loss(x, x, x, x, beta)\n\n\n@unittest.skipUnless(importlib.util.find_spec("peft") and importlib.util.find_spec("transformers"),\n                     "Tiny-model integration requires installed transformers and peft")\nclass TinyLlamaAdapterIntegrationTests(unittest.TestCase):\n    def test_sft_then_dpo_preserves_base_and_roundtrips_adapter(self):\n        from peft import LoraConfig, PeftModel, get_peft_model\n        from transformers import GenerationConfig, LlamaConfig, LlamaForCausalLM\n\n        torch.manual_seed(2026)\n        config = LlamaConfig(vocab_size=32, hidden_size=16, intermediate_size=32,\n                             num_hidden_layers=1, num_attention_heads=2,\n                             num_key_value_heads=2, max_position_embeddings=64,\n                             bos_token_id=1, eos_token_id=2, pad_token_id=0,\n                             attention_dropout=0.0)\n        base = LlamaForCausalLM(config)\n        base_initial = {k: v.detach().clone() for k, v in base.state_dict().items()}\n        model = get_peft_model(base, LoraConfig(r=2, lora_alpha=4, lora_dropout=0.0,\n                              target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM"))\n        frozen = {k: p.detach().clone() for k, p in model.named_parameters() if not p.requires_grad}\n        adapter_initial = {k: p.detach().clone() for k, p in model.named_parameters() if p.requires_grad}\n        self.assertTrue(adapter_initial)\n        self.assertTrue(all("lora_" in k for k in adapter_initial))\n        chosen_ids = torch.tensor([[1, 3, 5, 7, 9, 2]])\n        rejected_ids = torch.tensor([[1, 3, 5, 8, 10, 2]])\n        chosen_labels, rejected_labels = chosen_ids.clone(), rejected_ids.clone()\n        chosen_labels[:, :3] = rejected_labels[:, :3] = -100\n        mask = torch.ones_like(chosen_ids)\n        optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=0.01)\n        model.train()\n        loss = model(input_ids=chosen_ids, attention_mask=mask, labels=chosen_labels).loss\n        self.assertTrue(torch.isfinite(loss))\n        loss.backward()\n        optimizer.step()\n        optimizer.zero_grad(set_to_none=True)\n        self.assertTrue(any(not torch.equal(p, adapter_initial[k]) for k, p in model.named_parameters()\n                            if k in adapter_initial))\n\n        model.eval()  # Reference and DPO policy scores use dropout disabled.\n        with torch.no_grad():\n            ref_chosen = completion_logps(model(input_ids=chosen_ids, attention_mask=mask).logits, chosen_labels)\n            ref_rejected = completion_logps(model(input_ids=rejected_ids, attention_mask=mask).logits, rejected_labels)\n        sft_adapter = {k: p.detach().clone() for k, p in model.named_parameters() if p.requires_grad}\n        pi_chosen = completion_logps(model(input_ids=chosen_ids, attention_mask=mask).logits, chosen_labels)\n        pi_rejected = completion_logps(model(input_ids=rejected_ids, attention_mask=mask).logits, rejected_labels)\n        dpo = dpo_loss(pi_chosen, pi_rejected, ref_chosen, ref_rejected, 0.1)\n        self.assertAlmostEqual(dpo.item(), math.log(2), places=6)\n        dpo.backward()\n        optimizer.step()\n        self.assertTrue(any(not torch.equal(p, sft_adapter[k]) for k, p in model.named_parameters()\n                            if k in sft_adapter))\n        for k, p in model.named_parameters():\n            if k in frozen:\n                self.assertTrue(torch.equal(p, frozen[k]), k)\n        with torch.no_grad():\n            expected_logits = model(input_ids=chosen_ids, attention_mask=mask).logits\n        with tempfile.TemporaryDirectory() as directory:\n            model.save_pretrained(directory, safe_serialization=True)\n            restored_base = LlamaForCausalLM(config)\n            restored_base.load_state_dict(base_initial)\n            restored = PeftModel.from_pretrained(restored_base, directory).eval()\n            with torch.no_grad():\n                actual_logits = restored(input_ids=chosen_ids, attention_mask=mask).logits\n            torch.testing.assert_close(actual_logits, expected_logits, rtol=1e-5, atol=1e-6)\n            with torch.inference_mode():\n                generated = restored.generate(input_ids=chosen_ids[:, :3], attention_mask=mask[:, :3],\n                    generation_config=GenerationConfig(do_sample=False, max_new_tokens=2,\n                        bos_token_id=1, eos_token_id=2, pad_token_id=0))\n            self.assertTrue(torch.equal(generated[:, :3], chosen_ids[:, :3]))\n            self.assertGreater(generated.shape[1], 3)\n            self.assertLessEqual(generated.shape[1], 5)\n\n\nif __name__ == "__main__":\n    unittest.main()\n', 'tests/test_verifier.py': '"""Fabricated evidence used ONLY to test verification; never benchmark results.\n\nNo model is downloaded, trained, or used to produce this fixture. All responses\ncome from the deterministic rule engine, including those labeled as models.\n"""\nimport csv\nimport hashlib\nimport json\nimport math\nimport random\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nimport unittest\nfrom collections import Counter\nfrom pathlib import Path\n\nimport numpy as np\n\nPROJECT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(PROJECT))\nfrom scripts.verify_results import VerificationError, verify\nfrom support_data import CATEGORIES, build_dataset, rule_response, score_response, summarize, validate_dataset\n\nMETHODS = ("base", "sft", "dpo", "rule_engine")\nSPLITS = ("development", "test")\n\n\ndef write_json(path, value):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(value, indent=2) + "\\n")\n\n\ndef write_jsonl(path, rows):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text("".join(json.dumps(row) + "\\n" for row in rows))\n\n\ndef read_json(path):\n    return json.loads(path.read_text())\n\n\ndef sha(path):\n    return hashlib.sha256(path.read_bytes()).hexdigest()\n\n\ndef rehash(root):\n    write_json(root / "manifest_sha256.json", {\n        p.relative_to(root).as_posix(): sha(p) for p in sorted(root.rglob("*"))\n        if p.is_file() and p.name != "manifest_sha256.json" and p.relative_to(root).parts[0] not in ("adapters", "checkpoints")})\n\n\ndef fixture(root):\n    """Create internally consistent, explicitly fabricated evidence in temp space."""\n    config = read_json(PROJECT / "config.json")\n    config["bootstrap_resamples"] = 101\n    data = build_dataset(config["seed"])\n    write_json(root / "config.json", config)\n    write_json(root / "SYNTHETIC_TEST_FIXTURE.json", {"fabricated": True, "benchmark_result": False})\n    write_json(root / "run_status.json", {"status": "complete", "benchmark_completed": True,\n        "completed_stages": ["sft", "dpo"], "human_review_completed": False})\n    write_json(root / "protocol.json", {"methods": ["base", "sft", "dpo"], "reference_baseline": "rule_engine",\n        "selection": "None: fixed steps and unchanged decoding; no development or test tuning",\n        "primary_metric": "structured_pass", "test_evaluation_after_both_training_stages": True,\n        "max_new_tokens": config["max_new_tokens"], "do_sample": False, "adapter_merging": False,\n        "reply_semantic_quality_measured_by_primary_metric": False})\n    for name in ("environment.json", "parameters.json", "adapter_backups.json"):\n        write_json(root / name, {"SYNTHETIC_TEST_FIXTURE": True})\n    for name in ("train_evaluate.py", "support_data.py"):\n        path = root / "code" / name\n        path.parent.mkdir(exist_ok=True)\n        path.write_text("# Synthetic fixture only; no model execution.\\n")\n    for split, rows in data.items():\n        write_jsonl(root / "data" / (split + ".jsonl"), rows)\n    write_json(root / "data_audit.json", validate_dataset(data))\n    write_json(root / "data_manifest.json", {split: {"n": len(rows), "sha256": sha(root / "data" / (split + ".jsonl"))}\n        for split, rows in data.items()})\n    predictions = []\n    metrics = {}\n    for method in METHODS:\n        metrics[method] = {}\n        for split in SPLITS:\n            outputs = []\n            for row in data[split]:\n                response = rule_response(row)\n                outputs.append(response)\n                predictions.append({"method": method, "split": split, "id": row["id"], "category": row["category"],\n                    "response": response, "scores": score_response(row, response), "hit_token_limit": False,\n                    "generation_seconds": None if method == "rule_engine" else 0.01,\n                    "generated_tokens_including_eos": None if method == "rule_engine" else 10})\n            metrics[method][split] = summarize(data[split], outputs)\n    write_jsonl(root / "predictions.jsonl", predictions)\n    write_json(root / "metrics.json", metrics)\n    fields = ["method", "split"] + [key for key, value in metrics["base"]["test"].items() if not isinstance(value, dict)]\n    with (root / "metrics_summary.csv").open("w", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=fields); writer.writeheader()\n        for method in METHODS:\n            for split in SPLITS:\n                writer.writerow({"method": method, "split": split, **{key: metrics[method][split][key] for key in fields[2:]}})\n    write_json(root / "paired_comparisons.json", {name: {"mean_delta": 0.0, "ci95": [0.0, 0.0],\n        "b_better": 0, "a_better": 0, "ties": len(data["test"]), "resamples": config["bootstrap_resamples"],\n        "seed": config["seed"], "interpretation": "Descriptive paired case bootstrap; no multiplicity or domain-shift adjustment"}\n        for name in ("sft_minus_base", "dpo_minus_sft", "dpo_minus_base")})\n    train_ids = [row["id"] for row in data["train"]]\n    indices = np.random.default_rng(config["seed"] + 2).permutation(len(train_ids))[:config["dpo_examples"]]\n    dpo_rows = [data["train"][i] for i in indices]\n    dpo_ids = [row["id"] for row in dpo_rows]\n    write_json(root / "dpo_training_ids.json", dpo_ids)\n    write_json(root / "training_mixture.json", {\n        "sft_categories": dict(Counter(row["category"] for row in data["train"])),\n        "dpo_categories": dict(Counter(row["category"] for row in dpo_rows)),\n        "dpo_rejection_types": dict(Counter(row["rejection_type"] for row in dpo_rows)),\n        "subset_selection": "Fixed seeded random subset of training rows; no held-out information used"})\n    write_jsonl(root / "dpo_reference_logps.jsonl", [{"id": ticket, "chosen_logp": -10.0, "rejected_logp": -12.0} for ticket in dpo_ids])\n    write_json(root / "dpo_reference.json", {"sha256": sha(root / "dpo_reference_logps.jsonl"), "seconds": 0.1,\n        "reference": "Frozen SFT adapter on the same NF4 base", "reduction": "completion token sum including EOS"})\n    timings = {}\n    for stage, ids, seed in (("sft", train_ids, config["seed"]), ("dpo", dpo_ids, config["seed"] + 1)):\n        steps = config[stage + "_steps"]\n        count = steps * config["gradient_accumulation"]\n        write_jsonl(root / (stage + "_training_log.jsonl"), [{"step": step, "loss": math.log(2), "gradient_norm_before_clip": 1.0}\n            for step in range(1, steps + 1)])\n        rng = np.random.default_rng(seed)\n        order = [ids[i] for _ in range(math.ceil(count / len(ids))) for i in rng.permutation(len(ids))][:count]\n        write_json(root / (stage + "_training_order.json"), order)\n        timings[stage] = {"seconds": 0.1, "optimizer_steps": steps, "training_example_presentations": count,\n            "peak_allocated_gpu_bytes": 0, "peak_reserved_gpu_bytes": 0}\n    write_json(root / "timing.json", timings)\n    rng = random.Random(config["seed"])\n    selected = []\n    for category in sorted(CATEGORIES):\n        selected.extend(rng.sample(sorted(row["id"] for row in data["test"] if row["category"] == category), 2))\n    mapping = [{"review_id": f"R{100000000 + i}", "method": method, "id": ticket}\n        for i, (method, ticket) in enumerate((method, ticket) for method in METHODS[:3] for ticket in selected)]\n    write_json(root / "review_key.json", {"mapping": mapping})\n    row_map = {row["id"]: row for row in data["test"]}\n    with (root / "manual_review.csv").open("w", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=["review_id", "prompt", "response", "reply_factual", "instruction_following", "unsupported_advice", "notes"])\n        writer.writeheader()\n        for entry in mapping:\n            row = row_map[entry["id"]]\n            writer.writerow({"review_id": entry["review_id"], "prompt": row["prompt"], "response": rule_response(row)})\n    rehash(root)\n\n\nclass VerifierTests(unittest.TestCase):\n    @classmethod\n    def setUpClass(cls):\n        cls.template_temp = tempfile.TemporaryDirectory()\n        cls.template = Path(cls.template_temp.name)\n        fixture(cls.template)\n\n    @classmethod\n    def tearDownClass(cls):\n        cls.template_temp.cleanup()\n\n    def setUp(self):\n        self.temp = tempfile.TemporaryDirectory()\n        self.addCleanup(self.temp.cleanup)\n        self.root = Path(self.temp.name) / "synthetic_fixture"\n        shutil.copytree(self.template, self.root)\n\n    def edit_json(self, name, edit):\n        path = self.root / name\n        value = read_json(path); edit(value); write_json(path, value)\n\n    def rejected_after_rehash(self, message):\n        rehash(self.root)\n        with self.assertRaisesRegex(VerificationError, message):\n            verify(self.root)\n\n    def test_consistent_synthetic_fixture_and_blank_ratings_pass(self):\n        report = verify(self.root)\n        self.assertEqual(report["status"], "verified")\n        self.assertEqual(report["predictions"], 720)\n        self.assertEqual(report["review_rows_rated"], 0)\n\n    def test_cli_works_without_importing_torch(self):\n        command = \'\'\'import builtins, runpy, sys\noriginal_import = builtins.__import__\ndef checked_import(name, *args, **kwargs):\n    if name == "torch" or name.startswith("torch."):\n        raise AssertionError("Verifier imported torch")\n    return original_import(name, *args, **kwargs)\nbuiltins.__import__ = checked_import\nsys.argv = sys.argv[1:]\nrunpy.run_path(sys.argv[0], run_name="__main__")\n\'\'\'\n        result = subprocess.run([sys.executable, "-c", command, str(PROJECT / "scripts" / "verify_results.py"), str(self.root)], capture_output=True, text=True)\n        self.assertEqual(result.returncode, 0, result.stderr + result.stdout)\n        self.assertEqual(json.loads(result.stdout)["status"], "verified")\n        source = (PROJECT / "scripts" / "verify_results.py").read_text()\n        self.assertNotIn("import torch", source)\n        self.assertNotIn("from train_evaluate", source)\n\n    def test_manifest_detects_changed_file_without_rehash(self):\n        self.edit_json("metrics.json", lambda value: value["base"]["test"].update(structured_pass=0.0))\n        with self.assertRaisesRegex(VerificationError, "Manifest hash"):\n            verify(self.root)\n\n    def test_manifest_requires_complete_coverage(self):\n        self.edit_json("manifest_sha256.json", lambda value: value.pop("metrics.json"))\n        with self.assertRaisesRegex(VerificationError, "Manifest file coverage"):\n            verify(self.root)\n\n    def test_rehashed_flags_are_recomputed(self):\n        path = self.root / "predictions.jsonl"\n        rows = [json.loads(line) for line in path.read_text().splitlines()]\n        rows[0]["scores"]["structured_pass"] = False\n        write_jsonl(path, rows)\n        self.rejected_after_rehash("prediction scores")\n\n    def test_rehashed_metrics_are_recomputed(self):\n        self.edit_json("metrics.json", lambda value: value["base"]["test"].update(structured_pass=0.0))\n        self.rejected_after_rehash("metrics.base.test.structured_pass")\n\n    def test_rehashed_csv_is_recomputed(self):\n        path = self.root / "metrics_summary.csv"\n        with path.open(newline="") as stream:\n            reader = csv.DictReader(stream); fields = reader.fieldnames; rows = list(reader)\n        rows[0]["structured_pass"] = "0"\n        with path.open("w", newline="") as stream:\n            writer = csv.DictWriter(stream, fieldnames=fields); writer.writeheader(); writer.writerows(rows)\n        self.rejected_after_rehash("metric CSV")\n\n    def test_rehashed_bootstrap_is_recomputed(self):\n        self.edit_json("paired_comparisons.json", lambda value: value["sft_minus_base"].update(ci95=[-0.5, 0.5]))\n        self.rejected_after_rehash("paired_comparisons")\n\n    def test_duplicate_prediction_cannot_replace_missing_case(self):\n        path = self.root / "predictions.jsonl"\n        rows = [json.loads(line) for line in path.read_text().splitlines()]\n        rows[1] = rows[0].copy(); write_jsonl(path, rows)\n        self.rejected_after_rehash("duplicate prediction")\n\n    def test_training_order_cannot_use_heldout_id(self):\n        test_id = json.loads((self.root / "data/test.jsonl").read_text().splitlines()[0])["id"]\n        self.edit_json("sft_training_order.json", lambda value: value.__setitem__(0, test_id))\n        self.rejected_after_rehash("sft_training_order")\n\n    def test_reference_hash_is_checked_after_manifest_rehash(self):\n        self.edit_json("dpo_reference.json", lambda value: value.update(sha256="0" * 64))\n        self.rejected_after_rehash("DPO reference hash")\n\n    def test_reference_cannot_contain_nonfinite_scores(self):\n        path = self.root / "dpo_reference_logps.jsonl"\n        rows = [json.loads(line) for line in path.read_text().splitlines()]\n        rows[0]["chosen_logp"] = float("nan"); write_jsonl(path, rows)\n        self.rejected_after_rehash("Nonfinite JSON")\n\n    def test_review_key_cannot_duplicate_one_model_case(self):\n        self.edit_json("review_key.json", lambda value: value["mapping"].__setitem__(1, value["mapping"][0].copy()))\n        self.rejected_after_rehash("fixed balanced 12 tickets")\n\n    def test_interrupted_run_is_not_completed(self):\n        self.edit_json("run_status.json", lambda value: value.update(status="interrupted", benchmark_completed=False))\n        self.rejected_after_rehash("not a completed benchmark")\n\n\nif __name__ == "__main__":\n    unittest.main()\n', 'tests/test_stages.py': '"""Durable-stage tests with a fabricated CPU backend, never model benchmarks.\n\nEach stage runs in a fresh Python process. Only model/GPU operations are mocked;\nthe runner\'s checkpoints, guards, JSON persistence, exports and retry logic run.\n"""\nimport contextlib\nimport hashlib\nimport io\nimport json\nimport math\nimport os\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nimport traceback\nimport unittest\nfrom pathlib import Path\nfrom types import SimpleNamespace\nfrom unittest.mock import patch\n\nimport numpy as np\nimport torch\n\nPROJECT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(PROJECT))\nimport train_evaluate as runner\nfrom support_data import rule_response\n\n\ndef read(path):\n    return json.loads(Path(path).read_text())\n\n\ndef event(path, kind, **fields):\n    with Path(path).open("a") as stream:\n        stream.write(json.dumps({"kind": kind, **fields}) + "\\n")\n\n\ndef events(path):\n    return [json.loads(line) for line in Path(path).read_text().splitlines()] if Path(path).exists() else []\n\n\nclass FakeTokenizer:\n    eos_token_id = 2\n    bos_token_id = 1\n    eos_token = "</s>"\n    pad_token = "</s>"\n    padding_side = "right"\n\n    def apply_chat_template(self, messages, **kwargs):\n        return [1, 3, 4]\n\n    def __call__(self, text, **kwargs):\n        return {"input_ids": [5, 6]}\n\n    def save_pretrained(self, directory):\n        runner.write_json(Path(directory) / "tokenizer_config.json", {"fabricated_cpu_fixture": True})\n\n\nclass FakeCPUModel(torch.nn.Module):\n    """A scalar adapter state; not a language model or real safetensors file."""\n    def __init__(self, config, stage="base", value=0.0):\n        super().__init__()\n        self.lora_fixture = torch.nn.Parameter(torch.tensor(float(value)))\n        self.config = SimpleNamespace(use_cache=False)\n        self.peft_config = {"default": SimpleNamespace(revision=config["model_revision"])}\n        self.project_config = config\n        self.stage = stage\n\n    def get_nb_trainable_parameters(self):\n        return 1, 2\n\n    def save_pretrained(self, directory, **kwargs):\n        runner.write_json(Path(directory) / "adapter_model.safetensors", {\n            "fabricated_cpu_fixture": True, "value": self.lora_fixture.item(), "stage": self.stage})\n        runner.write_json(Path(directory) / "adapter_config.json", {\n            "base_model_name_or_path": self.project_config["model_id"],\n            "revision": self.project_config["model_revision"], "peft_type": "LORA", "r": 16,\n            "lora_alpha": 32, "lora_dropout": 0.0,\n            "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"]})\n\n\n@contextlib.contextmanager\ndef fake_backend(action, event_path, fail_after=None):\n    generated = 0\n\n    def reload_model(config, adapter_path=None):\n        if adapter_path is None:\n            model = FakeCPUModel(config)\n        else:\n            saved = read(Path(adapter_path) / "adapter_model.safetensors")\n            model = FakeCPUModel(config, saved["stage"], saved["value"])\n        event(event_path, "load_training", stage=model.stage, value=model.lora_fixture.item())\n        return model\n\n    def evaluation_model(config, method, run_dir):\n        if method == "base":\n            model = FakeCPUModel(config)\n        else:\n            saved = read(Path(run_dir) / "adapters" / method / "adapter_model.safetensors")\n            model = FakeCPUModel(config, saved["stage"], saved["value"])\n        event(event_path, "load_evaluation", method=method, value=model.lora_fixture.item())\n        return model\n\n    def train(model, tokenizer, encoded, rows, config, run_dir, stage, reference=None):\n        if action == "evaluate":\n            raise AssertionError("Evaluation or retry invoked training")\n        expected = 0.0 if stage == "sft" else 0.25\n        if model.lora_fixture.item() != expected:\n            raise AssertionError(f"{stage} loaded the wrong saved adapter state")\n        with torch.no_grad():\n            model.lora_fixture.add_(0.25 if stage == "sft" else 0.5)\n        model.stage = stage\n        event(event_path, "train", stage=stage, value=model.lora_fixture.item())\n        steps, accumulation = config[stage + "_steps"], config["gradient_accumulation"]\n        rng = np.random.default_rng(config["seed"] + int(stage == "dpo"))\n        count = steps * accumulation\n        order = [rows[i]["id"] for _ in range(math.ceil(count / len(rows))) for i in rng.permutation(len(rows))][:count]\n        runner.write_json(Path(run_dir) / (stage + "_training_order.json"), order)\n        runner.write_jsonl(Path(run_dir) / (stage + "_training_log.jsonl"), [\n            {"step": step, "loss": math.log(2), "gradient_norm_before_clip": 1.0}\n            for step in range(1, steps + 1)])\n        return {"seconds": 0.01, "optimizer_steps": steps, "training_example_presentations": count,\n                "peak_allocated_gpu_bytes": 0, "peak_reserved_gpu_bytes": 0}\n\n    def reference(model, encoded, rows, run_dir):\n        if model.stage != "sft" or model.lora_fixture.item() != 0.25:\n            raise AssertionError("Reference did not use the saved SFT state")\n        event(event_path, "reference", stage=model.stage)\n        records = [{"id": row["id"], "chosen_logp": -2.0, "rejected_logp": -3.0} for row in rows]\n        path = Path(run_dir) / "dpo_reference_logps.jsonl"\n        runner.write_jsonl(path, records)\n        return records, {"sha256": runner.digest(path), "seconds": 0.01,\n            "reference": "Frozen SFT adapter on the same NF4 base", "reduction": "completion token sum including EOS"}\n\n    def generate(model, tokenizer, row, config):\n        nonlocal generated\n        if fail_after is not None and generated >= abs(fail_after):\n            if fail_after < 0:\n                os._exit(9)  # Deliberately bypass runner exception/finally handlers.\n            raise RuntimeError("INTENTIONAL_TEST_EVALUATION_INTERRUPTION")\n        generated += 1\n        event(event_path, "generate", method=model.stage, split=row["split"], id=row["id"])\n        return rule_response(row), {"generation_seconds": 0.01,\n            "generated_tokens_including_eos": 10, "hit_token_limit": False}\n\n    with contextlib.ExitStack() as stack:\n        for name, replacement in {\n            "require_cuda": lambda: None,\n            "environment_info": lambda: {"fabricated_cpu_fixture": True, "gpu": "NONE", "cuda": None},\n            "load_tokenizer": lambda config: FakeTokenizer(),\n            "load_training_model": reload_model,\n            "load_evaluation_model": evaluation_model,\n            "training_loop": train,\n            "reference_scores": reference,\n            "generate": generate,\n        }.items():\n            stack.enter_context(patch.object(runner, name, replacement))\n        for name in ("empty_cache", "reset_peak_memory_stats", "synchronize"):\n            stack.enter_context(patch.object(torch.cuda, name, lambda *args, **kwargs: None))\n        stack.enter_context(patch.object(torch.cuda, "max_memory_allocated", lambda *args: 0))\n        stack.enter_context(patch.object(torch.cuda, "max_memory_reserved", lambda *args: 0))\n        if fail_after == -999:\n            stack.enter_context(patch.object(runner, "finalize_evidence", lambda *args, **kwargs: os._exit(9)))\n        yield\n\n\ndef worker(action, target, output_root, event_path, fail_after):\n    try:\n        with fake_backend(action, event_path, fail_after), contextlib.redirect_stdout(io.StringIO()):\n            if action == "sft":\n                result = runner.run_sft(target, output_root)\n            elif action == "dpo":\n                result = runner.run_dpo(target)\n            elif action == "evaluate":\n                result = runner.evaluate_saved(target)\n            else:\n                raise ValueError(action)\n        print(json.dumps({"ok": True, "run_dir": str(result)}))\n        return 0\n    except Exception as exc:\n        print(json.dumps({"ok": False, "error": str(exc), "type": type(exc).__name__}))\n        traceback.print_exc(file=sys.stderr)\n        return 2\n\n\ndef process(action, target, output_root, event_path, fail_after=None):\n    command = [sys.executable, str(Path(__file__).resolve()), "--worker", action, str(target),\n               str(output_root), str(event_path), "none" if fail_after is None else str(fail_after)]\n    result = subprocess.run(command, capture_output=True, text=True, timeout=90)\n    if result.returncode == 9 and not result.stdout:\n        return result, {"ok": False, "error": "INTENTIONAL_TEST_HARD_EXIT"}\n    try:\n        payload = json.loads(result.stdout)\n    except ValueError:\n        raise AssertionError(result.stdout + "\\n" + result.stderr)\n    return result, payload\n\n\nclass DurableStageTests(unittest.TestCase):\n    @classmethod\n    def setUpClass(cls):\n        cls.temp = tempfile.TemporaryDirectory(prefix="fabricated_stage_tests_")\n        cls.addClassCleanup(cls.temp.cleanup)\n        cls.template = Path(cls.temp.name)\n        config = read(PROJECT / "config.json")\n        config.update(sft_steps=2, dpo_steps=2, gradient_accumulation=1, dpo_examples=24, bootstrap_resamples=101)\n        runner.write_json(cls.template / "config.json", config)\n        cls.event_file = cls.template / "fake_backend_events.jsonl"\n        first, first_payload = process("sft", cls.template / "config.json", cls.template / "artifacts", cls.event_file)\n        if first.returncode:\n            raise AssertionError(first.stdout + first.stderr)\n        cls.run_name = Path(first_payload["run_dir"]).name\n        cls.run_dir = cls.template / "artifacts" / cls.run_name\n        cls.sft_status = read(cls.run_dir / "run_status.json")\n        cls.sft_timings = read(cls.run_dir / "timing.json")\n        second, _ = process("dpo", cls.run_dir, cls.template / "artifacts", cls.event_file)\n        if second.returncode:\n            raise AssertionError(second.stdout + second.stderr)\n\n    def setUp(self):\n        self.case_temp = tempfile.TemporaryDirectory(prefix="fabricated_stage_case_")\n        self.addCleanup(self.case_temp.cleanup)\n        self.root = Path(self.case_temp.name)\n        shutil.copytree(self.template, self.root, dirs_exist_ok=True)\n        self.run_dir = self.root / "artifacts" / self.run_name\n        self.event_file = self.root / "fake_backend_events.jsonl"\n\n    def run_worker(self, action, fail_after=None):\n        return process(action, self.run_dir, self.root / "artifacts", self.event_file, fail_after)\n\n    def test_stages_reconstruct_saved_state_in_fresh_processes(self):\n        self.assertEqual(self.sft_status["completed_stages"], ["sft"])\n        self.assertFalse(self.sft_status["benchmark_completed"])\n        self.assertEqual(self.sft_timings["sft"]["optimizer_steps"], 2)\n        status = read(self.run_dir / "run_status.json")\n        self.assertEqual(status["completed_stages"], ["sft", "dpo"])\n        self.assertFalse(status["benchmark_completed"])\n        self.assertEqual(read(self.run_dir / "timing.json")["dpo"]["optimizer_steps"], 2)\n        self.assertEqual(read(self.run_dir / "adapters/sft/adapter_model.safetensors")["value"], 0.25)\n        self.assertEqual(read(self.run_dir / "adapters/dpo/adapter_model.safetensors")["value"], 0.75)\n        history = events(self.event_file)\n        self.assertEqual([(e["stage"], e["value"]) for e in history if e["kind"] == "load_training"],\n                         [("base", 0.0), ("sft", 0.25)])\n        self.assertEqual([e["stage"] for e in history if e["kind"] == "train"], ["sft", "dpo"])\n        self.assertFalse((self.run_dir / "predictions.jsonl").exists())\n        self.assertEqual(set(read(self.run_dir / "adapter_backups.json")), {"sft", "dpo"})\n\n    def test_interrupted_evaluation_retries_only_missing_predictions_and_never_trains(self):\n        interrupted, payload = self.run_worker("evaluate", fail_after=7)\n        self.assertEqual(interrupted.returncode, 2, interrupted.stdout + interrupted.stderr)\n        self.assertIn("INTENTIONAL_TEST_EVALUATION_INTERRUPTION", payload["error"])\n        path = self.run_dir / "predictions.jsonl"\n        prior = [json.loads(line) for line in path.read_text().splitlines()]\n        self.assertEqual(len(prior), 7)\n        self.assertEqual(read(self.run_dir / "run_status.json")["completed_stages"], ["sft", "dpo"])\n        self.assertEqual(read(self.run_dir / "timing.json")["dpo"]["optimizer_steps"], 2)\n        resumed, result = self.run_worker("evaluate")\n        self.assertEqual(resumed.returncode, 0, resumed.stdout + resumed.stderr)\n        self.assertTrue(result["ok"])\n        records = [json.loads(line) for line in path.read_text().splitlines()]\n        keys = [(r["method"], r["split"], r["id"]) for r in records]\n        self.assertEqual(len(records), 720)\n        self.assertEqual(len(set(keys)), 720)\n        by_key = {key: row for key, row in zip(keys, records)}\n        for old in prior:\n            self.assertEqual(by_key[old["method"], old["split"], old["id"]], old)\n        generated = [(e["method"], e["split"], e["id"]) for e in events(self.event_file) if e["kind"] == "generate"]\n        self.assertEqual(len(generated), 540)\n        self.assertEqual(len(set(generated)), 540)\n        self.assertEqual([e["stage"] for e in events(self.event_file) if e["kind"] == "train"], ["sft", "dpo"])\n        self.assertTrue(read(self.run_dir / "run_status.json")["benchmark_completed"])\n        from scripts.verify_results import verify\n        self.assertEqual(verify(self.run_dir)["status"], "verified")\n        before = {p.relative_to(self.run_dir).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest()\n                  for p in self.run_dir.rglob("*") if p.is_file()}\n        previous_events = self.event_file.read_bytes()\n        again, again_payload = self.run_worker("evaluate")\n        self.assertEqual(again.returncode, 0, again.stdout + again.stderr)\n        self.assertTrue(again_payload["ok"])\n        after = {p.relative_to(self.run_dir).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest()\n                 for p in self.run_dir.rglob("*") if p.is_file()}\n        self.assertEqual(before, after)\n        self.assertEqual(previous_events, self.event_file.read_bytes())\n        archive = self.run_dir.parent / (self.run_dir.name + "_results.zip")\n        for corruption in ("missing", "truncated"):\n            with self.subTest(archive=corruption):\n                if corruption == "missing":\n                    archive.unlink()\n                else:\n                    archive.write_bytes(b"TRUNCATED ZIP TEST FIXTURE")\n                rejected, rejected_payload = self.run_worker("evaluate")\n                self.assertNotEqual(rejected.returncode, 0, rejected.stdout + rejected.stderr)\n                self.assertFalse(rejected_payload["ok"])\n                self.assertIn("Finalized evidence or its archive changed", rejected_payload["error"])\n                if corruption == "missing":\n                    self.assertFalse(archive.exists())\n                else:\n                    self.assertEqual(archive.read_bytes(), b"TRUNCATED ZIP TEST FIXTURE")\n                self.assertEqual(previous_events, self.event_file.read_bytes())\n                self.assertEqual(before, {p.relative_to(self.run_dir).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest()\n                    for p in self.run_dir.rglob("*") if p.is_file()})\n\n    def test_changed_config_is_rejected_before_model_loading(self):\n        config = read(self.run_dir / "config.json")\n        config["max_new_tokens"] += 1\n        runner.write_json(self.run_dir / "config.json", config)\n        before = self.event_file.read_bytes()\n        result, payload = self.run_worker("evaluate")\n        self.assertNotEqual(result.returncode, 0, result.stdout)\n        self.assertFalse(payload["ok"])\n        self.assertIn("config.json", payload["error"])\n        self.assertEqual(before, self.event_file.read_bytes())\n        self.assertFalse((self.run_dir / "predictions.jsonl").exists())\n\n    def test_hard_process_exit_reuses_durable_prefix_only(self):\n        killed, _ = self.run_worker("evaluate", fail_after=-13)\n        self.assertEqual(killed.returncode, 9, killed.stdout + killed.stderr)\n        path = self.run_dir / "predictions.jsonl"\n        durable = [json.loads(line) for line in path.read_text().splitlines()]\n        self.assertEqual(len(durable), 10)\n        resumed, _ = self.run_worker("evaluate")\n        self.assertEqual(resumed.returncode, 0, resumed.stdout + resumed.stderr)\n        final = [json.loads(line) for line in path.read_text().splitlines()]\n        self.assertEqual(final[:10], durable)\n        generated = [(e["method"], e["split"], e["id"]) for e in events(self.event_file) if e["kind"] == "generate"]\n        self.assertEqual(len(generated), 543)  # Three unsaved responses are retried.\n        self.assertEqual(len(set(generated)), 540)\n        for row in durable:\n            self.assertEqual(generated.count((row["method"], row["split"], row["id"])), 1)\n        self.assertEqual([e["stage"] for e in events(self.event_file) if e["kind"] == "train"], ["sft", "dpo"])\n\n    def test_hard_exit_before_final_export_recovers_without_generation(self):\n        killed, _ = self.run_worker("evaluate", fail_after=-999)\n        self.assertEqual(killed.returncode, 9, killed.stdout + killed.stderr)\n        predictions_path = self.run_dir / "predictions.jsonl"\n        prediction_bytes = predictions_path.read_bytes()\n        self.assertEqual(len(prediction_bytes.splitlines()), 720)\n        orphan = self.run_dir / ("metrics.json.tmp-" + "a" * 32)\n        orphan.write_text("INCOMPLETE ATOMIC WRITE TEST FIXTURE")\n        before_generations = [e for e in events(self.event_file) if e["kind"] == "generate"]\n        resumed, _ = self.run_worker("evaluate")\n        self.assertEqual(resumed.returncode, 0, resumed.stdout + resumed.stderr)\n        self.assertEqual(predictions_path.read_bytes(), prediction_bytes)\n        self.assertEqual([e for e in events(self.event_file) if e["kind"] == "generate"], before_generations)\n        self.assertEqual(len(before_generations), 540)\n        self.assertFalse(orphan.exists())\n        self.assertNotIn(orphan.name, read(self.run_dir / "manifest_sha256.json"))\n        self.assertTrue((self.run_dir.parent / (self.run_dir.name + "_results.zip")).is_file())\n        from scripts.verify_results import verify\n        self.assertEqual(verify(self.run_dir)["status"], "verified")\n\n    def test_corrupt_adapter_is_rejected_before_model_loading(self):\n        path = self.run_dir / "adapters/dpo/adapter_model.safetensors"\n        path.write_bytes(path.read_bytes() + b"CORRUPTION")\n        before = self.event_file.read_bytes()\n        result, payload = self.run_worker("evaluate")\n        self.assertNotEqual(result.returncode, 0, result.stdout)\n        self.assertFalse(payload["ok"])\n        self.assertIn("dpo adapter", payload["error"].lower())\n        self.assertEqual(before, self.event_file.read_bytes())\n\n\nif __name__ == "__main__":\n    if len(sys.argv) > 1 and sys.argv[1] == "--worker":\n        _, _, action, target, output_root, event_path, fail_after = sys.argv\n        raise SystemExit(worker(action, target, output_root, event_path,\n                               None if fail_after == "none" else int(fail_after)))\n    unittest.main()\n', 'README.md': "# Customer-Support Response Fine-Tuning: SFT vs DPO\n\nDoes supervised fine-tuning improve a small model's return-policy responses, and does preference optimization add value? This experiment compares a deterministic rule engine, the released TinyLlama chat model, SFT, and SFT followed by DPO on identical fictional support tickets.\n\n**Status: experiment prepared; a complete verified GPU benchmark is pending.** There are no model-quality results yet. Local checks and tiny CPU smoke runs check mechanics; they do not establish TinyLlama performance or validate the revised Drive workflow on a GPU.\n\n## Experiment\n\nEach ticket supplies a return window, elapsed days, receipt status, unused status, and damage status. The response must contain four JSON strings: `ticket_id`, `decision`, `reason`, and `reply`. Damage and missing receipts require review; other rules determine approval or denial.\n\n| Component | Fixed design |\n|---|---|\n| Model | `TinyLlama/TinyLlama-1.1B-Chat-v1.0` |\n| Revision | `fe8a4ea1ffedaf415f4da2f062534de366a451e6` |\n| Data | 480 training, 60 development, 120 test tickets; six balanced categories |\n| SFT | 120 steps; learning rate `2e-4`; gradient accumulation 4 |\n| DPO | 60 steps; learning rate `5e-5`; accumulation 4; beta 0.1; 240 fixed training pairs |\n| Adapter | LoRA rank 16, alpha 32, dropout 0; attention q/k/v/o projections |\n| Comparison | Same NF4 backbone, prompts, and decoding settings for all three model stages |\n\n“Base” means the released **chat** checkpoint before this project's adaptation. The dataset and chosen/rejected responses are synthetic; the preferences are programmatically specified, not collected from people. Fact combinations are disjoint across splits, excluding IDs from grouping. Wording templates are shared, so this tests new fact combinations rather than unfamiliar language.\n\nThe primary metric, `structured_pass`, checks the schema, ticket ID, decision, and reason. **It does not validate factuality of the free-text reply.** A response promising an unsupported payment can pass this metric. Review the blinded manual worksheet before making claims about response quality; exact agreement with a template is only a secondary diagnostic.\n\n## Run\n\n1. Open [Support_Response_FineTuning_Evaluation.ipynb](Support_Response_FineTuning_Evaluation.ipynb) in a fresh Colab T4 GPU session and run its setup cells. The notebook includes the project code and configuration.\n2. Section 6A mounts Google Drive. All run files are written under `My Drive/Support_Response_FineTuning/runs`. If runs already exist, select an exact folder name or explicitly choose a new run; the notebook never silently picks the newest one.\n3. Run SFT in 6B, DPO in 6C, and evaluation in 6D. Each training stage returns after its adapter and records are saved to Drive. Evaluation prints progress for 540 model generations and saves partial predictions. No automatic browser downloads occur during training or evaluation.\n4. Download the final evidence ZIP from Drive and save the executed notebook with its outputs. Complete a separate copy of the blinded review worksheet. Return the evidence ZIP and executed notebook for results verification.\n\nAfter a runtime reset, rerun setup, reconnect Drive, select the saved folder in 6A, and continue the needed stage. Completed stages are verified and reused. Evaluation resumes missing predictions; it cannot train models. Mid-training optimizer resume is not implemented. See [RECOVERY.md](RECOVERY.md) for exact limits.\n\nFor a local CUDA environment, install `requirements.txt`, then run:\n\n```bash\npython train_evaluate.py --config config.json --output-root artifacts\npython scripts/verify_results.py artifacts/<run_id>\n```\n\nFor checks without the full experiment:\n\n```bash\npython -m unittest discover -s tests -v\npython train_evaluate.py --smoke\n```\n\nKeep the protocol fixed before test evaluation. Report all model stages; do not choose a test winner, retune on those cases, and present the same test set as untouched. Dependency pins and model revision aid reproduction; GPU availability and runtime remain environment dependent.\n\n## Files and results\n\n| File | Purpose |\n|---|---|\n| `support_data.py` | Synthetic tickets, rule baseline, and scoring |\n| `train_evaluate.py`, `config.json` | Training and fixed comparison |\n| `scripts/verify_results.py` | Evidence consistency checks |\n| `tests/`, `validation/` | Mechanics checks and recorded preflight evidence |\n| `CODE_WALKTHROUGH.md` | Explain the code and trade-offs |\n| `BUSINESS_CASE.md` | Business baseline and evaluation decision |\n| `ATTRIBUTION.md`, `LICENSE` | Sources and license scope |\n| `RECOVERY.md` | Persistent saves, stage selection, and recovery limits |\n\nThe generated run folder contains the evidence needed to replace this pending status with measured results. Training loss alone is not a quality result. See [BUSINESS_CASE.md](BUSINESS_CASE.md) before treating a trained model as the business solution, and [GITHUB_UPLOAD.md](GITHUB_UPLOAD.md) for publication steps.\n", 'CODE_WALKTHROUGH.md': "# Code walkthrough\n\n`support_data.py` is the business contract. `rule_decision` applies rules in priority order; damage overrides missing receipts, which override deadline and unused-item checks. The deadline itself is included. `rule_response` turns that deterministic result into a complete response and supplies the non-model baseline.\n\n`build_dataset` samples reproducible cases from a finite pool. `fact_key` groups the actual policy inputs without ticket IDs, preventing a renamed copy of the same case from crossing splits. Categories are balanced. This controlled task makes errors interpretable, but its repeated language and simple rules limit generalization claims.\n\n`encode_completion` uses the tokenizer's chat template and masks prompt tokens with `-100`. SFT therefore learns the reference answer rather than predicting the supplied policy. Overlong examples fail instead of silently dropping facts or answer tokens. LoRA updates a small set of matrices while the quantized backbone stays frozen; that lowers training memory at the cost of a restricted update space.\n\nThe SFT adapter is saved before DPO. `run_sft` and `run_dpo` are separate stages. DPO reloads and continues **the same saved SFT adapter**, so its final checkpoint includes both stages' changes. There is no merge between stages. Each saved adapter loads against the same pinned original chat backbone; the final adapter must not be stacked onto the SFT adapter.\n\nBefore DPO updates, the runner computes and freezes the SFT model's chosen and rejected completion log probabilities for the 240 training pairs. `completion_logps` shifts logits for next-token prediction, ignores prompt/padding positions, and **sums** completion scores. Precomputing this fixed reference saves memory; it is valid for these fixed pairs and is not a changing reference policy.\n\n`dpo_loss` compares the policy's chosen-minus-rejected score with the frozen SFT reference difference, scaled by beta, using a negative log-sigmoid loss. It rewards the supplied preference relationship without a separate reward model. At the unchanged reference policy, the loss should be approximately `log(2)`. Lower loss does not guarantee better test responses.\n\n`score_response` rejects invalid JSON and duplicate keys, checks the exact four-field schema and reply length, then compares structured fields with the rule-derived answer. The `unsupported_reply` rejection intentionally exposes its boundary: correct fields can coexist with an invented payment promise. Blinded review assesses factual consistency and helpfulness beyond the automatic metric.\n\n`evaluate_saved` validates saved artifacts and runs inference without a training path. Progress and predictions are persisted during evaluation. On a retry, validated existing responses are retained and only missing cases are generated. Saved code, configuration, data, adapters, and environment constrain continuation so changed settings cannot silently produce a mixed comparison. Training timing is saved before evaluation begins.\n\nThe notebook writes run artifacts directly to mounted Drive. Browser downloads are an optional action after evaluation and are not used to establish that a stage was saved. This separates compute completion, persistent saves, and delivery to the user's computer.\n\nThe tests target token masking, score shifting/summing, DPO gradients, split isolation, rule boundaries, adapter save/reload, and staged interruption/retry behavior. A tiny randomly initialized CPU model checks the update path; mocked stage tests check orchestration. Neither establishes TinyLlama benchmark quality or validates Colab GPU/Drive behavior.\n", 'BUSINESS_CASE.md': '# Business application\n\nA support team needs consistent return decisions and clear customer responses. This project asks whether model adaptation improves response formatting and policy adherence under a controlled experiment. It does not yet demonstrate lower handling time, higher satisfaction, or financial return.\n\nFor the exact structured inputs used here, **a rule engine plus templates is the practical first option**. The rules already determine the answer completely. That baseline is transparent, cheap to run, and easy to update when the return policy changes. A model is not necessary to make this policy decision.\n\n| Option | What the experiment can show | Business consideration |\n|---|---|---|\n| Rules + templates | Correct application of the specified policy | Strong fit for structured tickets |\n| Released chat model | Starting formatting and policy errors | No project-specific training cost |\n| SFT | Effect of learning example responses | Adds training and version management |\n| SFT + DPO | Effect of synthetic preference pairs | Must justify its added complexity |\n\nEvaluate all stages on the fixed test set. Inspect failures by category, especially rule precedence and the inclusive deadline. Use the blinded worksheet to judge whether replies introduce promises or contradict facts. A high structured score cannot establish reliable free-text support.\n\nA useful next phase would add varied customer wording and missing or conflicting information, while leaving final policy decisions with the rule engine. Use separately collected, appropriately permitted examples, a new held-out set, and human review. Measure editing time, unsupported statements, escalation accuracy, latency, and operating cost before deciding whether a drafting model earns its place. No ROI estimate is claimed from this synthetic experiment.\n', 'ATTRIBUTION.md': "# Sources and license scope\n\nThis project's SFT-to-preference-tuning structure was informed by Chapter 12, “Fine-tuning Generation Models,” of *Hands-On Large Language Models* by Jay Alammar and Maarten Grootendorst (O'Reilly, 2024). See the [official repository and Chapter 12 notebook](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models/tree/main/chapter12). The uploaded source experiment records completed training; its logs are not results for this new support benchmark.\n\nMethod and model sources:\n\n- Hu et al., [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685), 2021.\n- Dettmers et al., [QLoRA: Efficient Finetuning of Quantized LLMs](https://arxiv.org/abs/2305.14314), 2023.\n- Rafailov et al., [Direct Preference Optimization: Your Language Model is Secretly a Reward Model](https://arxiv.org/abs/2305.18290), 2023.\n- TinyLlama team, [TinyLlama-1.1B-Chat-v1.0](https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0), pinned to [revision fe8a4ea](https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/tree/fe8a4ea1ffedaf415f4da2f062534de366a451e6).\n\nThe MIT license in this repository covers its original code, documentation, and synthetic dataset. It does not relicense model weights, third-party dependencies, or upstream materials. TinyLlama and the book's official code repository carry their own Apache-2.0 licenses. Model weights and the source PDF are not distributed in this project. The support cases describe a fictional store and contain no customer records.\n", 'GITHUB_UPLOAD.md': '# Publish the portfolio\n\nSuggested repository: `support-response-finetuning-evaluation`.\n\n1. Run `Support_Response_FineTuning_Evaluation.ipynb` on a fresh Colab T4 runtime. Mount Drive in section 6A; run SFT, DPO, and evaluation in their separate cells. Keep the executed notebook, final evidence ZIP, both adapter backups, and separate rated worksheet. Run files are saved under `My Drive/Support_Response_FineTuning/runs`.\n2. Verify the completed evidence with `python scripts/verify_results.py path/to/run_folder`. Review all method results and the written replies. Keep any failed or interrupted runs clearly identified; they are not completed benchmarks.\n3. Replace the pending-results paragraph in `README.md` with the measured comparison: structured success by method, paired differences, representative errors, manual-review counts, and observed training time/memory. State the single-seed and synthetic-data limits. Add the executed notebook and verified small evidence files under `results/`. Preserve the original evidence manifest; add human ratings as a separate file outside its run folder.\n4. Create the GitHub repository and upload the contents of this project folder. Use `.gitignore` to exclude weights, caches, and private backups. For browser uploads, select files deliberately because `.gitignore` does not filter files dragged into the browser. Do not upload adapter ZIPs, source PDFs, or the outer starter ZIP as a substitute for the project files.\n\nThe starter can be published earlier as an **experiment in progress**, retaining its pending status. Do not describe unexecuted model comparisons as measured results. The MIT license covers original project materials; see `ATTRIBUTION.md` for upstream license scope.\n', 'RECOVERY.md': '# Continue a saved experiment\n\nThe revised notebook saves directly under `My Drive/Support_Response_FineTuning/runs`. It does not depend on a browser download finishing while computation is running. A download progress bar is not confirmation that a file reached your computer.\n\n## New run\n\nUse a Colab T4 runtime, run the setup cells, and authorize the Drive mount in 6A. Leave `EXISTING_RUN_NAME` empty for a first run. If folders already exist, either select one or explicitly set `START_ANOTHER_RUN=True` to create an additional experiment. Nothing is deleted.\n\nRun 6B for SFT and wait for its saved message. Run 6C for DPO and wait for its saved message. Each completed stage has its adapter, tokenizer, hash manifest, and training records saved before evaluation. Run 6D for evaluation. It prints progress and checkpoints predictions; there are 540 model responses, so this stage can take longer than training.\n\n## Runtime disconnected during evaluation\n\n1. Reopen the same revised notebook in a fresh GPU session and run setup.\n2. Mount Drive in 6A. Copy the exact saved `generation_...` folder name into `EXISTING_RUN_NAME`, then rerun 6A.\n3. Run 6D. The runner checks saved configuration, source, data, adapters, environment, and predictions. It retains valid responses and generates missing cases without training.\n4. Continue the results, review, and verification cells. Download the evidence ZIP from Drive and save the executed notebook separately.\n\nDo not change code, model revision, dependencies, data, or decoding settings to bypass a continuation failure. Keep the original run and inspect the reported mismatch. This recovery path applies to runs made with this revised workflow; importing adapters from a different notebook requires separate provenance checks.\n\nA hard disconnect may lose up to nine responses since the last checkpoint; those responses are regenerated. Reported generation timings describe retained responses, not complete billed time including work lost before a checkpoint.\n\n## Runtime disconnected during training\n\nA completed SFT stage can be used to start DPO. A completed DPO stage can be evaluated. A stage stopped before its completion record cannot be treated as complete, even if an intermediate adapter exists. Snapshots do not include optimizer state, so automatic mid-stage training resume is not implemented. Keep the interrupted run and deliberately start a new run if a completed stage cannot be recovered.\n\n## Original temporary run folder is gone\n\nCheck completed downloads and any Drive backup made before the loss. Valid SFT/DPO adapter ZIPs can support a separately verified evaluation-only recovery, although unavailable training timings or logs must remain unavailable. Screenshots and progress bars cannot recreate model weights. If no saved adapters remain, a new training run is required; the revised workflow protects completed stages in Drive.\n\n## Download after completion\n\nOpen Google Drive, navigate to `Support_Response_FineTuning/runs`, and download the final `*_results.zip`. Adapter ZIPs sit beside the run folder. Human review files are inside the run folder; rate a copy outside the original evidence folder so its manifest stays intact. An optional standalone notebook cell can request the results ZIP through the browser after all computation finishes.\n', 'LICENSE': 'MIT License\n\nCopyright (c) 2026 Joe Williams\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n', '.gitignore': '__pycache__/\n*.pyc\n.ipynb_checkpoints/\nartifacts/\n.venv/\n.cache/\n*.safetensors\n*.bin\n'}
for name, source in PROJECT_FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source, encoding='utf-8')
os.chdir(PROJECT)
if str(PROJECT) not in sys.path: sys.path.insert(0, str(PROJECT))
for module_name in ('support_data', 'train_evaluate', 'scripts.verify_results'):
    sys.modules.pop(module_name, None)
importlib.invalidate_caches()
print('Project:', PROJECT)
print('Wrote', len(PROJECT_FILES), 'files. Full results are pending until training and evaluation finish.')

Project: /content/support_response_project
Wrote 17 files. Full results are pending until training and evaluation finish.


## 3. Inspect the policy and fixed dataset
Each fictional ticket contains a return window, elapsed days, receipt status, unused status, and damage status. Rules apply in priority order: damage → missing receipt → expired window → used item → approval. The deadline day is included.

There are **480 training, 60 development, and 120 test cases**, balanced across six policy categories. Actual fact tuples, excluding IDs, are disjoint across splits. Wording templates are shared: this is a controlled test of new fact combinations, not a test of unseen customer language.

All chosen responses and preferences are synthetic. Rejected responses cover incorrect decisions, reasons, IDs, extra fields, markdown, and unsupported promises. No held-out rows are used for gradient updates. Development rows are reported for diagnosis; no model selection or tuning is performed in this fixed run.

In [5]:
from support_data import build_dataset, validate_dataset, POLICY, rule_response
from train_evaluate import write_jsonl
CONFIG = json.loads(Path('config.json').read_text())
DATA = build_dataset(CONFIG['seed'])
Path('data').mkdir(exist_ok=True)
for split, rows in DATA.items(): write_jsonl(Path('data') / f'{split}.jsonl', rows)
print(json.dumps(validate_dataset(DATA), indent=2))
print('\nPOLICY\n' + POLICY)
example = DATA['train'][0]
print('\nTraining example:\n' + example['prompt'])
print('\nChosen:\n' + example['completion'])
print('\nRejected (' + example['rejection_type'] + '):\n' + example['rejected'])

{
  "rows": {
    "train": 480,
    "development": 60,
    "test": 120
  },
  "unique_fact_groups": 660,
  "shared_fact_groups": 0,
  "synthetic": true,
  "wording_policy": "Shared fixed templates; new fact combinations, not unseen language."
}

POLICY
You draft responses for a fictional store. Use only the supplied facts.
Apply these rules in this exact order:
1. If damaged is true, decision=review and reason=damaged_item, regardless of other facts.
2. Otherwise, if receipt is false, decision=review and reason=missing_receipt.
3. Otherwise, if elapsed_days > return_window_days, decision=deny and reason=outside_window.
4. Otherwise, if unused is false, decision=deny and reason=used_item.
5. Otherwise decision=approve and reason=eligible. The deadline day is included.
Return one JSON object only with exactly four string fields: ticket_id, decision, reason, reply.
Copy ticket_id exactly. Write a helpful reply of 1 to 60 words consistent with the decision.
Do not add markdown, extra keys,

## 4. Understand the update and evaluation contract
- **SFT:** completion-only next-token loss, 120 optimizer steps × 4 examples; LoRA rank 16 on attention q/k/v/o projections. The NF4-quantized backbone stays frozen.
- **DPO:** continue the same SFT adapter for 60 steps × 4 examples, using 240 fixed training pairs. Freeze the SFT reference log probabilities before updating. The saved DPO adapter contains both stages; load it directly onto the original pinned backbone.
- **Generation:** greedy decoding with the same 128-token budget for every model stage. Evaluate only after both training stages finish. Reject overlong inputs instead of truncating silently.
- **Primary metric:** `structured_pass` verifies JSON schema, ticket ID, decision, and reason. It does **not** establish the factuality or usefulness of the free-text reply. Exact agreement with a reference reply is a secondary diagnostic.
- **Uncertainty:** paired bootstrap intervals describe variation across these test cases for one training seed. They do not measure variation across training seeds or real deployments.

The deterministic baseline is expected to satisfy these rules by construction. Its purpose is to identify whether a learned model is needed at all. Training loss alone is not evidence of business value.

In [6]:
import inspect
from train_evaluate import encode_completion, completion_logps, dpo_loss
from support_data import score_response
for fn in [encode_completion, completion_logps, dpo_loss, score_response]:
    print(inspect.getsource(fn))
print('Fixed configuration:\n', json.dumps(CONFIG, indent=2))

def encode_completion(tokenizer, prompt, completion, max_length):
    prompt_ids = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=True, add_generation_prompt=True)
    response_ids = tokenizer(completion, add_special_tokens=False)["input_ids"]
    if tokenizer.eos_token_id is None:
        raise ValueError("Tokenizer needs an EOS token")
    if not response_ids or response_ids[-1] != tokenizer.eos_token_id:
        response_ids.append(tokenizer.eos_token_id)
    ids = list(prompt_ids) + response_ids
    if not prompt_ids or len(ids) > max_length:
        raise ValueError(f"Sequence length {len(ids)} exceeds limit {max_length}; no truncation performed")
    return {"input_ids": ids, "attention_mask": [1] * len(ids),
            "labels": [-100] * len(prompt_ids) + response_ids}

def completion_logps(logits, labels):
    """Sum causal next-token log probabilities; prompt/pad labels are -100."""
    if logits.ndim != 3 or labels.shape != logits.shap

## 5. Run the local checks and token-length preflight
The CPU test uses a tiny randomly initialized Llama to check SFT, DPO, gradient flow, and adapter save/reload. It is not a quality test of TinyLlama. The evidence verifier is also tested using explicitly fabricated fixture responses. No fixture scores are experiment results.

In [7]:
subprocess.check_call([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'])
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_id'], revision=CONFIG['model_revision'], trust_remote_code=False)
length_report = {}
for split, rows in DATA.items():
    lengths = [len(encode_completion(tokenizer, r['prompt'], r[field], CONFIG['max_length'])['input_ids'])
               for r in rows for field in ('completion', 'rejected')]
    prompt_lengths = [len(tokenizer.apply_chat_template([{'role':'user','content':r['prompt']}],
                      tokenize=True, add_generation_prompt=True)) for r in rows]
    assert max(prompt_lengths) + CONFIG['max_new_tokens'] <= CONFIG['max_length']
    length_report[split] = {'min_pair_length':min(lengths), 'max_pair_length':max(lengths),
                           'max_generation_budget':max(prompt_lengths)+CONFIG['max_new_tokens']}
print(json.dumps(length_report, indent=2))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

{
  "train": {
    "min_pair_length": 345,
    "max_pair_length": 367,
    "max_generation_budget": 427
  },
  "development": {
    "min_pair_length": 344,
    "max_pair_length": 367,
    "max_generation_budget": 426
  },
  "test": {
    "min_pair_length": 345,
    "max_pair_length": 369,
    "max_generation_budget": 427
  }
}


## 6A. Connect Drive and choose a run
The run folder, training logs, adapter weights, and results are written directly inside **My Drive → Support_Response_FineTuning → runs**. Colab will ask you to authorize Drive. This notebook stops if the mount is unavailable; it does not silently fall back to temporary storage.

For your first run, leave `EXISTING_RUN_NAME` empty and `START_ANOTHER_RUN=False`. If runs already exist, their names are printed: paste the intended folder name into `EXISTING_RUN_NAME` to continue it. To deliberately start an additional experiment, leave the name empty and set `START_ANOTHER_RUN=True`. Existing runs are never deleted or silently selected.

After a runtime disconnect, reopen this revised notebook, run setup again, and select the exact saved folder. Completed stages are skipped after their saved artifacts are validated. A partially completed training stage does not have optimizer-state resume; the notebook explains when a new run is necessary. Evaluation can resume saved predictions without retraining.

In [9]:
from pathlib import Path
import json
from google.colab import drive
from IPython.display import display, FileLink
drive.mount('/content/drive', force_remount=True, timeout_ms=300000)
DRIVE_HOME = Path('/content/drive/MyDrive')
if not DRIVE_HOME.is_dir():
    raise RuntimeError('Drive is not mounted. Complete Drive authorization before training.')
OUTPUT_ROOT = DRIVE_HOME / 'Support_Response_FineTuning' / 'runs'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

EXISTING_RUN_NAME = 'generation_20261125T164816Z-6faa0a'
START_ANOTHER_RUN = False

existing = sorted(p for p in OUTPUT_ROOT.glob('generation_*') if p.is_dir())
for p in existing:
    status_path = p / 'run_status.json'
    status = json.loads(status_path.read_text()) if status_path.exists() else {}
    print(p.name, '| status:', status.get('status', 'unavailable'),
          '| completed training:', status.get('completed_stages', []))
if EXISTING_RUN_NAME:
    if Path(EXISTING_RUN_NAME).name != EXISTING_RUN_NAME:
        raise ValueError('Enter the folder name only, without slashes.')
    RUN_DIR = OUTPUT_ROOT / EXISTING_RUN_NAME
    if RUN_DIR not in existing:
        raise ValueError('That exact run folder was not found in Drive.')
    print('Selected existing run:', RUN_DIR)
elif existing and not START_ANOTHER_RUN:
    raise RuntimeError('Existing runs found. Select one above, or deliberately set START_ANOTHER_RUN=True.')
else:
    RUN_DIR = None
    print('A new run will be created by the SFT cell. Nothing has trained yet.')
print('Persistent output folder:', OUTPUT_ROOT)

Mounted at /content/drive
generation_20261125T164816Z-6faa0a | status: complete | completed training: ['sft', 'dpo']
generation_20261125T195025Z-932ad3 | status: running | completed training: ['sft', 'dpo']
Selected existing run: /content/drive/MyDrive/Support_Response_FineTuning/runs/generation_20261125T164816Z-6faa0a
Persistent output folder: /content/drive/MyDrive/Support_Response_FineTuning/runs


## 6B. Train and save SFT
Run this cell and wait for its **SFT saved in Drive** message. It returns before DPO or evaluation begins. For a selected existing run, the runner validates the completed SFT artifacts rather than retraining. A new run name is printed as soon as the run is created; keep that name if the cell is interrupted.

In [ ]:
from train_evaluate import run_sft, run_dpo, evaluate_saved, validate_saved_run
if RUN_DIR is None:
    RUN_DIR = run_sft('config.json', output_root=OUTPUT_ROOT)
else:
    validate_saved_run(RUN_DIR, required_stages=('sft',))
    print('Using verified saved SFT; no training repeated.')
print('SFT saved in Drive:', RUN_DIR)
print('SFT adapter ZIP:', RUN_DIR.parent / (RUN_DIR.name + '_sft_adapter.zip'))

## 6C. Continue from SFT and save DPO
This loads the completed SFT adapter and trains DPO with the fixed protocol. Both stage adapters and training records remain in Drive. Re-running this cell validates and skips an already completed DPO stage. It never starts SFT again.

In [ ]:
if RUN_DIR is None:
    raise RuntimeError('Run or select a completed SFT stage first.')
RUN_DIR = run_dpo(RUN_DIR)
print('DPO saved in Drive:', RUN_DIR)
print('DPO adapter ZIP:', RUN_DIR.parent / (RUN_DIR.name + '_dpo_adapter.zip'))

## 6D. Evaluate saved models
This cell performs **540 model generations** (180 cases × three model stages), plus the rule baseline. It can take substantially longer than the training loss display suggests. Progress is printed and saved during evaluation; silence from browser download widgets is no longer used as a progress signal.

If evaluation stops, reconnect Drive, select this exact run in 6A, and run this evaluation cell again after setup. Valid saved predictions are reused; missing responses are generated. Model weights, decoding, data, and code must still match the saved run. This cell cannot train either stage.

In [11]:
from train_evaluate import evaluate_saved
if RUN_DIR is None:
    raise RuntimeError('Select a run with both completed training stages first.')
RUN_DIR = evaluate_saved(RUN_DIR)
RESULTS_ZIP = RUN_DIR.parent / (RUN_DIR.name + '_results.zip')
print('Evaluation complete. Results saved in Drive:', RESULTS_ZIP)
print('Human review is still pending.')

Already complete; unchanged: /content/drive/MyDrive/Support_Response_FineTuning/runs/generation_20261125T164816Z-6faa0a
Evaluation complete. Results saved in Drive: /content/drive/MyDrive/Support_Response_FineTuning/runs/generation_20261125T164816Z-6faa0a_results.zip
Human review is still pending.


## 7. Inspect measured results
Report every method. A lower DPO training loss can coexist with worse held-out performance. Compare structured success, JSON validity, category failures, token-limit hits, and generation time. The base checkpoint is already chat-tuned; this is incremental task adaptation, not pretraining from scratch.

In [12]:
import csv, html
from IPython.display import HTML
if RUN_DIR is None or not (RUN_DIR / 'metrics_summary.csv').is_file():
    raise RuntimeError('Complete or resume evaluation in 6D before inspecting results.')
with (RUN_DIR / 'metrics_summary.csv').open() as f: table = list(csv.DictReader(f))
columns = ['method','split','n_examples','json_valid','schema_valid','structured_pass','reference_reply_exact']
header = ''.join('<th>'+html.escape(k)+'</th>' for k in columns)
body = ''.join('<tr>'+''.join('<td>'+html.escape(str(row[k]))+'</td>' for k in columns)+'</tr>' for row in table)
display(HTML('<table><thead><tr>'+header+'</tr></thead><tbody>'+body+'</tbody></table>'))
print('\nPaired test comparisons:\n' + (RUN_DIR / 'paired_comparisons.json').read_text())
print('\nTraining and evaluation timing:\n' + (RUN_DIR / 'timing.json').read_text())
predictions = [json.loads(line) for line in (RUN_DIR / 'predictions.jsonl').read_text().splitlines()]
for method in ['base','sft','dpo']:
    rows = [p for p in predictions if p['method']==method and p['split']=='test']
    times = sorted(p['generation_seconds'] for p in rows)
    print(method, 'test token-limit hits:', sum(p['hit_token_limit'] for p in rows),
          '| mean seconds/response:', round(sum(times)/len(times),3))

method,split,n_examples,json_valid,schema_valid,structured_pass,reference_reply_exact
base,development,60,0.0,0.0,0.0,0.0
base,test,120,0.0,0.0,0.0,0.0
sft,development,60,1.0,1.0,0.7166666666666667,0.7166666666666667
sft,test,120,1.0,1.0,0.725,0.725
dpo,development,60,0.85,0.5333333333333333,0.25,0.11666666666666667
dpo,test,120,0.9,0.6166666666666667,0.30833333333333335,0.08333333333333333
rule_engine,development,60,1.0,1.0,1.0,1.0
rule_engine,test,120,1.0,1.0,1.0,1.0



Paired test comparisons:
{
  "sft_minus_base": {
    "mean_delta": 0.725,
    "ci95": [
      0.6416666666666667,
      0.8
    ],
    "b_better": 87,
    "a_better": 0,
    "ties": 33,
    "resamples": 2000,
    "seed": 2026,
    "interpretation": "Descriptive paired case bootstrap; no multiplicity or domain-shift adjustment"
  },
  "dpo_minus_sft": {
    "mean_delta": -0.4166666666666667,
    "ci95": [
      -0.5333333333333333,
      -0.3
    ],
    "b_better": 11,
    "a_better": 61,
    "ties": 48,
    "resamples": 2000,
    "seed": 2026,
    "interpretation": "Descriptive paired case bootstrap; no multiplicity or domain-shift adjustment"
  },
  "dpo_minus_base": {
    "mean_delta": 0.30833333333333335,
    "ci95": [
      0.225,
      0.4
    ],
    "b_better": 37,
    "a_better": 0,
    "ties": 83,
    "resamples": 2000,
    "seed": 2026,
    "interpretation": "Descriptive paired case bootstrap; no multiplicity or domain-shift adjustment"
  }
}


Training and evaluation timing:

## 8. Review the written replies before unblinding
The worksheet contains the same 12 test cases (two per category) for each of the three model stages: **36 shuffled responses**. Review IDs conceal the method in the worksheet; do not open `review_key.json` until ratings are complete. This is a small descriptive review, not a blinded clinical-style study or a production acceptance test.

Save a separate copy named `manual_review_rated.csv`; keep the original evidence intact. For each response, record:

| Column | Rating rule |
|---|---|
| `reply_factual` | 1 if the reply is consistent with the supplied facts and policy; 0 otherwise, including no usable reply |
| `instruction_following` | 1 if it follows the requested format, length, and decision instructions; 0 otherwise |
| `unsupported_advice` | 1 if it invents a promise, payment, action, or policy; 0 otherwise |
| `notes` | Explain the error and whether the reply is clear and useful; flag missing or unusable text |

Open the key after rating, join on `review_id`, and report counts by method with examples. An unavailable response must not appear better merely because it contains no unsupported advice. A single reviewer and small sample limit the conclusions.

In [13]:
print('Worksheet saved in Drive:', RUN_DIR / 'manual_review.csv')
display(FileLink(str(RUN_DIR / 'manual_review.csv')))
print('Rate a separate copy before opening review_key.json.')

Worksheet saved in Drive: /content/drive/MyDrive/Support_Response_FineTuning/runs/generation_20261125T164816Z-6faa0a/manual_review.csv


/content/drive/MyDrive/Support_Response_FineTuning/runs/generation_20261125T164816Z-6faa0a/manual_review.csv

Rate a separate copy before opening review_key.json.


## 9. Verify and export
The verifier recomputes scores and metrics from saved outputs and checks data separation, coverage, hashes, and review mapping. It checks consistency; it cannot prove that model inference actually occurred or replace rerunning the experiment.

Open **Google Drive → My Drive → Support_Response_FineTuning → runs** to download the results ZIP and keep both adapter ZIPs. The review CSV is inside the corresponding run folder. Use **File → Download → Download .ipynb** in Colab to save this notebook with its actual outputs. Return the evidence ZIP and executed notebook for portfolio verification, plus the separate rated worksheet when ready. The GitHub starter remains marked pending until those results are reviewed.

The optional browser-download cell below is disabled by default and runs only after evaluation. It is not part of saving the run to Drive.

In [14]:
subprocess.check_call([sys.executable, 'scripts/verify_results.py', str(RUN_DIR)])
print('Verified results ZIP in Drive:', RESULTS_ZIP)
display(FileLink(str(RESULTS_ZIP)))

Verified results ZIP in Drive: /content/drive/MyDrive/Support_Response_FineTuning/runs/generation_20261125T164816Z-6faa0a_results.zip


/content/drive/MyDrive/Support_Response_FineTuning/runs/generation_20261125T164816Z-6faa0a_results.zip

In [15]:
DOWNLOAD_RESULTS_IN_BROWSER = False
if DOWNLOAD_RESULTS_IN_BROWSER:
    from google.colab import files
    files.download(str(RESULTS_ZIP))

## 10. Explain the result and business trade-offs
Complete these after the run and manual review:

1. How much did SFT change held-out structured success? Did DPO add improvement or regress?
2. Which categories failed? Were errors about JSON, copying IDs, policy precedence, or the deadline boundary?
3. Did correct structured fields coexist with incorrect replies? Cite examples from the rated worksheet.
4. What did the additional training stage cost in measured GPU time and memory? How did output length affect latency?
5. Given that rules plus templates already solve these structured tickets, where would a model add value? A plausible next experiment is extracting facts from real customer wording, while keeping policy decisions deterministic and routing uncertain cases to staff.

Do not infer staffing savings, customer satisfaction, production safety, or general language understanding from this synthetic benchmark. Additional test seeds and representative business data would be separate experiments.

## Sources
Inspired by Chapter 12, “Fine-tuning Generation Models,” of *Hands-On Large Language Models* by Jay Alammar and Maarten Grootendorst ([official code](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models/tree/main/chapter12)).

[TinyLlama model and license](https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0) · [LoRA](https://arxiv.org/abs/2106.09685) · [QLoRA](https://arxiv.org/abs/2305.14314) · [DPO](https://arxiv.org/abs/2305.18290).

Original project code, documentation, and synthetic data use the included MIT license. Upstream models, dependencies, and source materials retain their own licenses. No model weights or source PDF are included.